In [67]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent

print("Текущая папка:", Path.cwd())
print("Корень проекта:", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Текущая папка: c:\temp\shift_ml\notebooks
Корень проекта: C:\temp\shift_ml


In [69]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

from src.feature_engineering import (
    prepare_features,
    add_missingness_and_ev_features,
    add_age_mileage_ratio_features,
    add_title_hierarchy_features,
)

ImportError: cannot import name 'add_title_hierarchy_features' from 'src.feature_engineering' (C:\temp\shift_ml\src\feature_engineering.py)

In [70]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(exist_ok=True)

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

print("train:", train.shape)
print("X_test:", X_test.shape)

train: (8340, 23)
X_test: (8341, 22)


In [5]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100

In [71]:
import importlib
import src.feature_engineering as fe

importlib.reload(fe)

from src.feature_engineering import (
    prepare_features,
    add_missingness_and_ev_features,
    prepare_features,
    add_title_features,
)

In [8]:
X_base = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_v2 = add_missingness_and_ev_features(X_base)

y = train[TARGET_COLUMN].copy()

print("X_base:", X_base.shape)
print("X_v2:", X_v2.shape)
print("y:", y.shape)

X_base: (8340, 29)
X_v2: (8340, 39)
y: (8340,)


In [9]:
target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
)

train_idx, valid_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=42,
    stratify=target_bins,
)

print("Train objects:", len(train_idx))
print("Validation objects:", len(valid_idx))

Train objects: 6672
Validation objects: 1668


In [10]:
X_train_v2_split = X_v2.iloc[train_idx].copy()
X_valid_v2_split = X_v2.iloc[valid_idx].copy()

y_train_split = y.iloc[train_idx].copy()
y_valid_split = y.iloc[valid_idx].copy()

Список признаков

In [11]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

feature_columns_v2 = [
    column
    for column in X_v2.columns
    if column not in EXCLUDED_COLUMNS
]

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

new_numeric_columns = [
    "Пробег_число_пропущен",
    "Двигатель_объём_л_пропущен",
    "Двигатель_цилиндры_пропущен",
    "Расход_л_на_100км_пропущен",
    "Электромобиль",
    "Нулевой_расход_у_не_EV",
    "Нулевой_объём_у_не_EV",
    "Расход_л_на_100км_очищенный",
    "Двигатель_объём_л_очищенный",
    "Количество_пропусков_техданных",
]

numeric_columns_v2 = base_numeric_columns + new_numeric_columns

categorical_columns_v2 = [
    column
    for column in feature_columns_v2
    if column not in numeric_columns_v2
]

print("Всего признаков:", len(feature_columns_v2))
print("Числовых:", len(numeric_columns_v2))
print("Категориальных:", len(categorical_columns_v2))

assert set(numeric_columns_v2).issubset(feature_columns_v2)
assert set(numeric_columns_v2 + categorical_columns_v2) == set(feature_columns_v2)

Всего признаков: 32
Числовых: 19
Категориальных: 13


Подготовка матрицы для CatBoost

In [12]:
X_model_v2 = X_v2[feature_columns_v2].copy()

for column in categorical_columns_v2:
    X_model_v2[column] = (
        X_model_v2[column]
        .fillna("__MISSING__")
        .astype(str)
    )

X_train_v2_split = X_model_v2.iloc[train_idx].copy()
X_valid_v2_split = X_model_v2.iloc[valid_idx].copy()

y_train_split = y.iloc[train_idx].copy()
y_valid_split = y.iloc[valid_idx].copy()

assert X_train_v2_split.index.equals(y_train_split.index)
assert X_valid_v2_split.index.equals(y_valid_split.index)

print(X_train_v2_split.shape)
print(X_valid_v2_split.shape)

(6672, 32)
(1668, 32)


Контроль новых фичей

In [13]:
new_feature_check = X_model_v2[
    [
        "Пробег_число_пропущен",
        "Двигатель_объём_л_пропущен",
        "Двигатель_цилиндры_пропущен",
        "Расход_л_на_100км_пропущен",
        "Электромобиль",
        "Нулевой_расход_у_не_EV",
        "Нулевой_объём_у_не_EV",
        "Количество_пропусков_техданных",
    ]
].sum()

display(new_feature_check)

Пробег_число_пропущен              274
Двигатель_объём_л_пропущен         801
Двигатель_цилиндры_пропущен        862
Расход_л_на_100км_пропущен         812
Электромобиль                       66
Нулевой_расход_у_не_EV             126
Нулевой_объём_у_не_EV                6
Количество_пропусков_техданных    2749
dtype: int64

Обучение CatBoost

In [14]:
from catboost import CatBoostRegressor

catboost_missing_ev_v2 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_missing_ev_v2.fit(
    X_train_v2_split,
    np.log1p(y_train_split),
    cat_features=categorical_columns_v2,
    eval_set=(
        X_valid_v2_split,
        np.log1p(y_valid_split),
    ),
    use_best_model=True,
    early_stopping_rounds=200,
)

0:	learn: 0.6528268	test: 0.6504997	best: 0.6504997 (0)	total: 250ms	remaining: 12m 30s
300:	learn: 0.1836733	test: 0.2160628	best: 0.2160628 (300)	total: 31.6s	remaining: 4m 43s
600:	learn: 0.1488900	test: 0.2030710	best: 0.2030684 (597)	total: 1m 3s	remaining: 4m 13s
900:	learn: 0.1282760	test: 0.1976278	best: 0.1975892 (892)	total: 1m 34s	remaining: 3m 40s
1200:	learn: 0.1114124	test: 0.1947945	best: 0.1947915 (1199)	total: 2m 6s	remaining: 3m 9s
1500:	learn: 0.0984487	test: 0.1932240	best: 0.1932229 (1495)	total: 2m 37s	remaining: 2m 37s
1800:	learn: 0.0883219	test: 0.1924395	best: 0.1924306 (1786)	total: 3m 9s	remaining: 2m 6s
2100:	learn: 0.0790903	test: 0.1914191	best: 0.1914096 (2092)	total: 3m 41s	remaining: 1m 34s
2400:	learn: 0.0715389	test: 0.1907981	best: 0.1907981 (2400)	total: 4m 13s	remaining: 1m 3s
2700:	learn: 0.0650979	test: 0.1903608	best: 0.1903410 (2697)	total: 4m 45s	remaining: 31.6s
2999:	learn: 0.0590366	test: 0.1902397	best: 0.1902180 (2968)	total: 5m 17s	rema

CatBoostRegressor(allow_writing_files=False, depth=8, iterations=3000, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=300)

Оценка

In [15]:
y_pred_valid_v2 = np.maximum(
    np.expm1(
        catboost_missing_ev_v2.predict(X_valid_v2_split)
    ),
    1,
)

missing_ev_v2_result = pd.DataFrame(
    {
        "model": [
            "CatBoost log1p + missingness and EV features"
        ],
        "best_iteration": [
            catboost_missing_ev_v2.get_best_iteration()
        ],
        "validation_mape_pct": [
            round(
                mape_percent(
                    y_valid_split,
                    y_pred_valid_v2,
                ),
                3,
            )
        ],
    }
)

comparison = pd.DataFrame(
    {
        "model": [
            "CatBoost log1p, base features",
            "CatBoost log1p, missingness + EV v2",
        ],
        "validation_mape_pct": [
            13.433,
            missing_ev_v2_result.loc[
                0,
                "validation_mape_pct",
            ],
        ],
    }
).sort_values("validation_mape_pct").reset_index(drop=True)

display(missing_ev_v2_result)
display(comparison)

,model,best_iteration,validation_mape_pct
0,CatBoost log1p + missingness and EV features,2968,13.41


,model,validation_mape_pct
0,"CatBoost log1p, missingness + EV v2",13.410
1,"CatBoost log1p, base features",13.433


гипотеза была хорошей, но результат говорит следующее:

CatBoost и так умеет использовать NaN;

исходные Топливо, Двигатель, Расход уже позволяют ему распознавать EV и технические особенности;

добавленные флаги не дали нового сильного сигнала.

Пакет v2 пока не принимаем как улучшение. Сохраняем код, но базовой моделью остаётся CatBoost без него.

Теперь логично исследовать связку:

Год выпуска + Пробег

Сам пробег недостаточен:

100 000 км у машины 2010 года
≠
100 000 км у машины 2023 года

Потенциально полезные признаки:

возраст автомобиля;

log1p(пробег);

пробег в год;

log1p(пробег в год).

# ============================================================
# Experiment v3:
# year + mileage interaction features
# ============================================================

In [5]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

In [6]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100

In [9]:
TARGET_COLUMN = "Цена"

# Базовые признаки: очистка текстовых пропусков + парсинг.
X_base = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

# ВАЖНО:
# Здесь только v3.
# Мы НЕ вызываем add_missingness_and_ev_features.
X_v3 = add_age_mileage_ratio_features(X_base)

y = train[TARGET_COLUMN].copy()

print("X_base:", X_base.shape)
print("X_v3:", X_v3.shape)
print("y:", y.shape)

X_base: (8340, 29)
X_v3: (8340, 32)
y: (8340,)


In [10]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

feature_columns_v3 = [
    column
    for column in X_v3.columns
    if column not in EXCLUDED_COLUMNS
]

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

# Только новые числовые признаки v3.
new_numeric_columns_v3 = [
    "Пробег_на_год",
    "Авто_до_года",
]

numeric_columns_v3 = (
    base_numeric_columns
    + new_numeric_columns_v3
)

categorical_columns_v3 = [
    column
    for column in feature_columns_v3
    if column not in numeric_columns_v3
]

print("Всего признаков:", len(feature_columns_v3))
print("Числовых:", len(numeric_columns_v3))
print("Категориальных:", len(categorical_columns_v3))

print("\nНовые признаки:")
print(new_numeric_columns_v3)
print("Годовой_сегмент — категориальный признак.")

assert set(numeric_columns_v3).issubset(feature_columns_v3)
assert (
    set(numeric_columns_v3 + categorical_columns_v3)
    == set(feature_columns_v3)
)

Всего признаков: 25
Числовых: 11
Категориальных: 14

Новые признаки:
['Пробег_на_год', 'Авто_до_года']
Годовой_сегмент — категориальный признак.


In [11]:
X_model_v3 = X_v3[feature_columns_v3].copy()

# CatBoost требует строкового типа для категорий.
for column in categorical_columns_v3:
    X_model_v3[column] = (
        X_model_v3[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert "car_id" not in X_model_v3.columns
assert "Предложение" not in X_model_v3.columns

display(
    X_model_v3[
        [
            "Год выпуска",
            "Пробег_число",
            "Пробег_на_год",
            "Авто_до_года",
            "Годовой_сегмент",
        ]
    ].head(10)
)

,Год выпуска,Пробег_число,Пробег_на_год,Авто_до_года,Годовой_сегмент
0,2022.0,244,122.0,0,2020_2022
1,2014.0,202408,20240.8,0,2010_2015
2,2021.0,23301,7767.0,0,2020_2022
3,2022.0,17359,8679.5,0,2020_2022
4,2018.0,100456,16742.666667,0,2015_2018
5,2019.0,58762,11752.4,0,2018_2020
6,2021.0,16691,5563.666667,0,2020_2022
7,2022.0,18407,9203.5,0,2020_2022
8,2019.0,97790,19558.0,0,2018_2020
9,2004.0,203910,10195.5,0,2000_2005


восстанавливаем тот же самый holdout, что использовали раньше:

In [12]:
target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
)

train_idx, valid_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=42,
    stratify=target_bins,
)

X_train_v3_split = X_model_v3.iloc[train_idx].copy()
X_valid_v3_split = X_model_v3.iloc[valid_idx].copy()

y_train_split = y.iloc[train_idx].copy()
y_valid_split = y.iloc[valid_idx].copy()

print("Train:", X_train_v3_split.shape)
print("Validation:", X_valid_v3_split.shape)

assert X_train_v3_split.index.equals(y_train_split.index)
assert X_valid_v3_split.index.equals(y_valid_split.index)

Train: (6672, 25)
Validation: (1668, 25)


Обучаем CatBoost с теми же параметрами:

In [13]:
catboost_age_mileage_v3 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_age_mileage_v3.fit(
    X_train_v3_split,
    np.log1p(y_train_split),
    cat_features=categorical_columns_v3,
    eval_set=(
        X_valid_v3_split,
        np.log1p(y_valid_split),
    ),
    use_best_model=True,
    early_stopping_rounds=200,
)

0:	learn: 0.6523978	test: 0.6507523	best: 0.6507523 (0)	total: 255ms	remaining: 12m 45s
300:	learn: 0.1839695	test: 0.2187474	best: 0.2187474 (300)	total: 27.8s	remaining: 4m 9s
600:	learn: 0.1460339	test: 0.2051367	best: 0.2051367 (600)	total: 58.5s	remaining: 3m 53s
900:	learn: 0.1229041	test: 0.1988430	best: 0.1988430 (900)	total: 1m 29s	remaining: 3m 27s
1200:	learn: 0.1049636	test: 0.1959401	best: 0.1959387 (1199)	total: 2m	remaining: 3m
1500:	learn: 0.0919096	test: 0.1938682	best: 0.1938439 (1495)	total: 2m 31s	remaining: 2m 31s
1800:	learn: 0.0813560	test: 0.1928952	best: 0.1928580 (1792)	total: 3m 3s	remaining: 2m 1s
2100:	learn: 0.0725600	test: 0.1923183	best: 0.1922813 (2090)	total: 3m 34s	remaining: 1m 31s
2400:	learn: 0.0654604	test: 0.1918050	best: 0.1917914 (2379)	total: 4m 5s	remaining: 1m 1s
2700:	learn: 0.0585508	test: 0.1915593	best: 0.1915411 (2662)	total: 4m 39s	remaining: 31s
2999:	learn: 0.0530949	test: 0.1913466	best: 0.1913301 (2904)	total: 5m 12s	remaining: 0us

CatBoostRegressor(allow_writing_files=False, depth=8, iterations=3000, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=300)

Считаем результат:

In [14]:
y_pred_valid_v3 = np.maximum(
    np.expm1(
        catboost_age_mileage_v3.predict(X_valid_v3_split)
    ),
    1,
)

age_mileage_v3_result = pd.DataFrame(
    {
        "model": [
            "CatBoost log1p + mileage per year + year segment"
        ],
        "best_iteration": [
            catboost_age_mileage_v3.get_best_iteration()
        ],
        "validation_mape_pct": [
            round(
                mape_percent(
                    y_valid_split,
                    y_pred_valid_v3,
                ),
                3,
            )
        ],
    }
)

comparison_v3 = pd.DataFrame(
    {
        "model": [
            "CatBoost log1p, base features",
            "CatBoost log1p, missingness + EV v2",
            "CatBoost log1p, age + mileage v3",
        ],
        "validation_mape_pct": [
            13.433,
            13.410,
            age_mileage_v3_result.loc[
                0,
                "validation_mape_pct",
            ],
        ],
    }
).sort_values(
    "validation_mape_pct"
).reset_index(drop=True)

display(age_mileage_v3_result)
display(comparison_v3)

,model,best_iteration,validation_mape_pct
0,CatBoost log1p + mileage per year + year segment,2904,13.407


,model,validation_mape_pct
0,"CatBoost log1p, age + mileage v3",13.407
1,"CatBoost log1p, missingness + EV v2",13.410
2,"CatBoost log1p, base features",13.433


Что именно проверяет v3
Пробег_на_год

Отличает, например:

машина 2010 года, 100 000 км
→ около 7 000–8 000 км в год

машина 2023 года, 100 000 км
→ около 100 000 км в год

Следующий перспективный блок — не числовые дубликаты, а структура категорий и полного названия:

Бренд + Модель
Бренд + Тип кузова
Модель + годовой сегмент
Полное название без года
токены комплектации: SPORT, GT, TURBO, HYBRID, AWD, 4X4 и т.д.

Почему это перспективнее: цена автомобиля сильно зависит не просто от марки и года, а от конкретной модели, версии и комплектации. CatBoost видит Полное название как отдельную категорию, но не всегда умеет хорошо обобщать похожие комплектации между разными годами.

# ============================================================
# EDA: категориальные признаки и потенциальные взаимодействия
# ============================================================

In [17]:
TARGET_COLUMN = "Цена"

X_eda = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

eda = X_eda.copy()
eda[TARGET_COLUMN] = train[TARGET_COLUMN].to_numpy()

print(eda.shape)
display(eda.head())

(8340, 30)


,car_id,Бренд,Год выпуска,Модель,Тип машины,Полное название,Исползование,КПП,Двигатель,Привод,...,Количество владельцев,Предложение,Пробег_число,Расход_л_на_100км,Двигатель_цилиндры,Двигатель_объём_л,Двери_число,Кресла_число,Штат,Цена
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,GWM,2022.0,HAVAL,SPRINGWOOD GWM HAVAL,2022 GWM HAVAL H6 ULTRA AWD,DEMO,Automatic,"4 cyl, 2 L",AWD,...,5.0,32145.0,244,9.8,4,2.0,4,5,QLD,38812
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,FORD,2014.0,TERRITORY,SUV,2014 FORD TERRITORY TITANIUM (4X4),USED,Automatic,"6 cyl, 2.7 L",AWD,...,6.0,15949.0,202408,9.0,6,2.7,4,7,NSW,15950
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,NISSAN,2021.0,X-TRAIL,SUV,2021 NISSAN X-TRAIL TI (4WD),USED,Automatic,"4 cyl, 2.5 L",4WD,...,7.0,39616.0,23301,8.3,4,2.5,4,5,VIC,41990
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,VOLVO,2022.0,XC60,SUV,2022 VOLVO XC60 B5 MOMENTUM MHEV,USED,Automatic,"4 cyl, 2 L",AWD,...,9.0,70449.0,17359,7.6,4,2.0,4,5,WA,69900
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,RENAULT,2018.0,KOLEOS,SUV,2018 RENAULT KOLEOS INTENS (4X4),USED,Automatic,"4 cyl, 2 L",4WD,...,8.0,22895.0,100456,6.1,4,2.0,4,5,VIC,27950


In [18]:
categorical_eda_columns = [
    "Бренд",
    "Модель",
    "Тип машины",
    "Тип кузова",
    "Топливо",
    "КПП",
    "Привод",
    "Штат",
    "Полное название",
]

category_summary = []

for column in categorical_eda_columns:
    category_summary.append(
        {
            "feature": column,
            "unique_values": eda[column].nunique(dropna=False),
            "missing_pct": round(eda[column].isna().mean() * 100, 2),
            "min_group_size": int(
                eda.groupby(column, dropna=False)["Цена"].size().min()
            ),
            "median_group_size": float(
                eda.groupby(column, dropna=False)["Цена"].size().median()
            ),
            "max_group_size": int(
                eda.groupby(column, dropna=False)["Цена"].size().max()
            ),
        }
    )

category_summary = pd.DataFrame(category_summary)

display(category_summary)

,feature,unique_values,missing_pct,min_group_size,median_group_size,max_group_size
0,Бренд,66,0.00,1,14.0,1426
1,Модель,630,0.00,1,3.0,222
2,Тип машины,429,0.16,1,1.0,2969
3,Тип кузова,11,1.59,9,300.0,3440
4,Топливо,9,3.82,1,319.0,3463
5,КПП,3,1.47,123,980.0,7237
6,Привод,5,0.00,546,1532.0,3476
7,Штат,9,2.55,37,412.0,3226
8,Полное название,5434,0.00,1,1.0,30


отдельно посмотри самые частые комбинации марки и модели:

In [19]:
brand_model_report = (
    eda.assign(
        Бренд_Модель=(
            eda["Бренд"].astype("string")
            + " | "
            + eda["Модель"].astype("string")
        )
    )
    .groupby("Бренд_Модель", dropna=False)
    .agg(
        cars=("Цена", "size"),
        median_price=("Цена", "median"),
        mean_price=("Цена", "mean"),
        price_std=("Цена", "std"),
        median_year=("Год выпуска", "median"),
        median_mileage=("Пробег_число", "median"),
    )
    .sort_values("cars", ascending=False)
    .reset_index()
)

display(brand_model_report.head(30))

,Бренд_Модель,cars,median_price,mean_price,price_std,median_year,median_mileage
0,TOYOTA | HILUX,222,36373.0,37796.279279,17246.640580,2017.0,144935.5
1,TOYOTA | COROLLA,200,28939.0,26424.535000,10160.551411,2018.0,64705.0
2,FORD | RANGER,200,35994.5,39895.530000,19875.141141,2017.0,132243.5
3,HYUNDAI | I30,198,23640.0,24003.070707,9830.475892,2019.0,57282.5
4,TOYOTA | LANDCRUISER,195,67988.0,70169.794872,40107.747894,2017.0,118041.0
5,TOYOTA | RAV4,171,44888.0,41510.719298,16964.537730,2020.0,61141.0
6,MITSUBISHI | TRITON,164,31994.5,32459.719512,14742.741108,2017.0,115000.0
7,HOLDEN | COMMODORE,155,19990.0,22121.103226,12659.410707,2012.0,159420.0
8,NISSAN | X-TRAIL,143,26990.0,25834.734266,10208.449131,2018.0,85974.5
9,NISSAN | NAVARA,125,25950.0,28179.992000,13934.853879,2015.0,165781.0


И проверим, насколько сильно цена меняется внутри одной и той же модели:

In [20]:
model_price_variability = (
    eda.groupby(["Бренд", "Модель"], dropna=False)
    .agg(
        cars=("Цена", "size"),
        median_price=("Цена", "median"),
        min_price=("Цена", "min"),
        max_price=("Цена", "max"),
        price_std=("Цена", "std"),
        year_min=("Год выпуска", "min"),
        year_max=("Год выпуска", "max"),
    )
    .query("cars >= 10")
    .sort_values("price_std", ascending=False)
    .reset_index()
)

display(model_price_variability.head(30))

,Бренд,Модель,cars,median_price,min_price,max_price,price_std,year_min,year_max
0,PORSCHE,CAYENNE,17,52800.0,18890,229900,77086.774530,2003.0,2023.0
1,LAND,ROVER,120,59800.0,8995,299900,57472.309923,2001.0,2023.0
2,BMW,X5,27,44999.0,11999,219888,49470.498196,2006.0,2023.0
3,MERCEDES-BENZ,C63,10,113439.0,49980,203888,45030.494172,2011.0,2021.0
4,TOYOTA,LANDCRUISER,195,67988.0,9990,299900,40107.747894,1991.0,2023.0
5,TOYOTA,LAND,15,79950.0,27999,147990,39781.779299,1996.0,2023.0
6,VOLKSWAGEN,TOUAREG,12,75440.0,18913,123888,39647.296610,2011.0,2022.0
7,FORD,MUSTANG,28,59990.0,29990,234990,36552.005924,1997.0,2022.0
8,AUDI,Q7,20,35915.0,4990,129990,35085.275471,2007.0,2022.0
9,NISSAN,PATROL,32,29994.5,9999,97995,31379.426388,1998.0,2022.0


1. Что видно

Ты нашёл очень важную картину.

Низкая кардинальность — устойчивые признаки
Тип кузова: 11 значений
Топливо: 9
КПП: 3
Привод: 5
Штат: 9

Это хорошие категории: по ним много объектов в каждой группе, CatBoost может надёжно оценивать их влияние.

Высокая кардинальность — риск разреженности
Модель: 630 значений, медианный размер группы 3
Тип машины: 429 значений, медианный размер группы 1
Полное название: 5434 значений, медианный размер группы 1

Полное название почти уникально: у половины названий всего один объект. В сыром виде CatBoost не может хорошо обобщать цену между похожими комплектациями: для него это часто просто разные редкие строки.

Ключевое наблюдение: Модель может быть неполной

Примеры:

LAND | ROVER
JEEP | GRAND
HYUNDAI | SANTA
TOYOTA | LAND

Похоже, поле Модель иногда хранит только первый токен многословной модели:

LAND ROVER
GRAND CHEROKEE
SANTA FE
LAND CRUISER

Это объясняет широкую вариативность цены внутри некоторых групп. Например, LAND | ROVER объединяет много разных автомобилей, а не одну модель.

Самая перспективная ветка сейчас — не добавлять ещё один числовой ratio, а структурировать Полное название.

Не нужно сразу делать огромный NLP-пайплайн. Сначала выделим переносимые признаки:

название без года
бренд + модель
бренд + тип кузова
многословные токены комплектации
признаки turbo / sport / hybrid / AWD / 4X4 / GT / luxury
число токенов и цифр в названии

Логика:

2022 AUDI E-TRON RS GT QUATTRO
2023 AUDI E-TRON RS GT QUATTRO

Это разные сырые строки из-за года, но похожая комплектация. Если убрать год и выделить RS, GT, QUATTRO, модель сможет обобщать их связь с ценой.

In [21]:
import re

In [22]:
title_eda = eda.copy()

title_eda["Название_верхний"] = (
    title_eda["Полное название"]
    .astype("string")
    .str.upper()
    .str.strip()
)

title_eda["Название_без_года"] = (
    title_eda["Название_верхний"]
    .str.replace(r"^\d{4}\s+", "", regex=True)
)

title_eda["Название_число_слов"] = (
    title_eda["Название_без_года"]
    .str.split()
    .str.len()
)

title_eda["Название_есть_год"] = (
    title_eda["Название_верхний"]
    .str.contains(r"^\d{4}\b", regex=True, na=False)
    .astype("int8")
)

display(
    title_eda[
        [
            "Бренд",
            "Модель",
            "Полное название",
            "Название_без_года",
            "Название_число_слов",
            "Цена",
        ]
    ].sample(30, random_state=42)
)

,Бренд,Модель,Полное название,Название_без_года,Название_число_слов,Цена
7470,PORSCHE,PANAMERA,2015 PORSCHE PANAMERA DIESEL,PORSCHE PANAMERA DIESEL,3,129888
586,LEXUS,UX200,2023 LEXUS UX200 LUXURY,LEXUS UX200 LUXURY,3,46085
7961,TOYOTA,COROLLA,2018 TOYOTA COROLLA SX (HYBRID),TOYOTA COROLLA SX (HYBRID),4,34990
4405,BMW,220I,2020 BMW 220I M SPORT GRAN COUPE,BMW 220I M SPORT GRAN COUPE,6,54990
5363,VOLKSWAGEN,TIGUAN,2018 VOLKSWAGEN TIGUAN 110 TDI COMFORTLINE,VOLKSWAGEN TIGUAN 110 TDI COMFORTLINE,5,33930
1201,AUDI,Q3,2014 AUDI Q3 2.0 TDI QUATTRO (130KW),AUDI Q3 2.0 TDI QUATTRO (130KW),6,27995
7649,TOYOTA,HILUX,2019 TOYOTA HILUX SR (4X4),TOYOTA HILUX SR (4X4),4,45999
879,HOLDEN,COMMODORE,2013 HOLDEN COMMODORE SS-V REDLINE,HOLDEN COMMODORE SS-V REDLINE,4,39990
5885,LAND,ROVER,2019 LAND ROVER DISCOVERY SPORT TD4 (132KW) HS...,LAND ROVER DISCOVERY SPORT TD4 (132KW) HSE AWD,8,49900
6495,HYUNDAI,TUCSON,2016 HYUNDAI TUCSON ACTIVE (FWD),HYUNDAI TUCSON ACTIVE (FWD),4,23999


Теперь посмотри наиболее частые названия после удаления года:

In [23]:
title_without_year_report = (
    title_eda
    .groupby("Название_без_года", dropna=False)
    .agg(
        cars=("Цена", "size"),
        median_price=("Цена", "median"),
        min_year=("Год выпуска", "min"),
        max_year=("Год выпуска", "max"),
        price_std=("Цена", "std"),
    )
    .sort_values("cars", ascending=False)
    .reset_index()
)

display(title_without_year_report.head(40))

,Название_без_года,cars,median_price,min_year,max_year,price_std
0,TOYOTA HILUX SR (4X4),89,39990.0,2005.0,2023.0,13153.265934
1,HYUNDAI I30 ACTIVE,88,23244.5,2012.0,2022.0,3318.927301
2,KIA CERATO S,61,23977.0,2014.0,2022.0,4327.017693
3,TOYOTA LANDCRUISER PRADO GXL (4X4),49,37800.0,1998.0,2020.0,18143.415434
4,TOYOTA HIACE LWB,47,25999.0,2007.0,2022.0,6785.010647
5,HOLDEN COMMODORE SV6,43,15985.0,2005.0,2017.0,6730.967269
6,TOYOTA COROLLA ASCENT,41,14995.0,2002.0,2019.0,6653.213302
7,TOYOTA HILUX WORKMATE,40,22470.0,1999.0,2021.0,6868.026147
8,TOYOTA HILUX SR5 (4X4),39,46990.0,2005.0,2021.0,14971.982590
9,MITSUBISHI TRITON GLX (4X4),38,25924.5,2008.0,2023.0,11415.711380


И отдельно соберём частые токены. Это покажет, какие слова реально встречаются в комплектациях:

In [24]:
from collections import Counter

tokens = (
    title_eda["Название_без_года"]
    .dropna()
    .str.findall(r"[A-Z0-9]+")
    .explode()
)

token_counts = (
    tokens
    .value_counts()
    .rename_axis("token")
    .reset_index(name="count")
)

# Убираем слишком общие и малоинформативные токены.
common_tokens_to_exclude = {
    "AUTO",
    "AUTOMATIC",
    "MANUAL",
    "USED",
}

token_counts = token_counts.loc[
    ~token_counts["token"].isin(common_tokens_to_exclude)
]

display(token_counts.head(100))

,token,count
0,4X4,1573
1,TOYOTA,1426
2,HYUNDAI,618
3,2,600
4,MAZDA,590
...,...,...
96,RENAULT,83
97,SPORTAGE,82
98,CROSS,80
99,RANGE,80


Сначала сделаем компактный и чистый пакет v4:

Название_без_года — категориальный;
Название_число_слов — числовой;
Название_есть_4X4;
Название_есть_AWD;
Название_есть_TURBO;
Название_есть_SPORT;
Название_есть_HYBRID;
Название_есть_GT;
Название_есть_LUXURY;
Название_есть_DIESEL.

In [29]:
X_base = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_v4 = add_title_features(X_base)

y = train[TARGET_COLUMN].copy()

print("X_base:", X_base.shape)
print("X_v4:", X_v4.shape)

X_base: (8340, 29)
X_v4: (8340, 39)


Создай v4

In [30]:
X_base = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_v4 = add_title_features(X_base)

y = train[TARGET_COLUMN].copy()

print("X_base:", X_base.shape)
print("X_v4:", X_v4.shape)

X_base: (8340, 29)
X_v4: (8340, 39)


Полный блок списков признаков для v4

In [31]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

feature_columns_v4 = [
    column
    for column in X_v4.columns
    if column not in EXCLUDED_COLUMNS
]

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

title_numeric_columns = [
    "Название_число_слов",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
]

numeric_columns_v4 = (
    base_numeric_columns
    + title_numeric_columns
)

categorical_columns_v4 = [
    column
    for column in feature_columns_v4
    if column not in numeric_columns_v4
]

print("Всего:", len(feature_columns_v4))
print("Числовых:", len(numeric_columns_v4))
print("Категориальных:", len(categorical_columns_v4))

assert "Название_без_года" in categorical_columns_v4
assert "car_id" not in feature_columns_v4
assert "Предложение" not in feature_columns_v4

Всего: 32
Числовых: 18
Категориальных: 14


Подготовка данных и обучение

In [32]:
X_model_v4 = X_v4[feature_columns_v4].copy()

for column in categorical_columns_v4:
    X_model_v4[column] = (
        X_model_v4[column]
        .fillna("__MISSING__")
        .astype(str)
    )

X_train_v4_split = X_model_v4.iloc[train_idx].copy()
X_valid_v4_split = X_model_v4.iloc[valid_idx].copy()

y_train_split = y.iloc[train_idx].copy()
y_valid_split = y.iloc[valid_idx].copy()
catboost_title_v4 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_title_v4.fit(
    X_train_v4_split,
    np.log1p(y_train_split),
    cat_features=categorical_columns_v4,
    eval_set=(
        X_valid_v4_split,
        np.log1p(y_valid_split),
    ),
    use_best_model=True,
    early_stopping_rounds=200,
)

0:	learn: 0.6531358	test: 0.6514630	best: 0.6514630 (0)	total: 96.6ms	remaining: 4m 49s
300:	learn: 0.1808574	test: 0.2149168	best: 0.2149168 (300)	total: 27.9s	remaining: 4m 10s
600:	learn: 0.1445260	test: 0.2016736	best: 0.2016736 (600)	total: 58.6s	remaining: 3m 54s
900:	learn: 0.1241643	test: 0.1961008	best: 0.1961008 (900)	total: 1m 44s	remaining: 4m 2s
1200:	learn: 0.1075204	test: 0.1934240	best: 0.1934240 (1200)	total: 2m 16s	remaining: 3m 24s
1500:	learn: 0.0957797	test: 0.1919253	best: 0.1919158 (1491)	total: 2m 48s	remaining: 2m 47s
1800:	learn: 0.0852123	test: 0.1908033	best: 0.1907840 (1797)	total: 3m 20s	remaining: 2m 13s
2100:	learn: 0.0763610	test: 0.1898777	best: 0.1898777 (2100)	total: 3m 52s	remaining: 1m 39s
2400:	learn: 0.0685299	test: 0.1894474	best: 0.1894438 (2390)	total: 4m 26s	remaining: 1m 6s
2700:	learn: 0.0617877	test: 0.1892922	best: 0.1892525 (2572)	total: 5m 7s	remaining: 34.1s
2999:	learn: 0.0558758	test: 0.1890919	best: 0.1890919 (2999)	total: 5m 38s	re

CatBoostRegressor(allow_writing_files=False, depth=8, iterations=3000, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=300)

In [33]:
y_pred_valid_v4 = np.maximum(
    np.expm1(
        catboost_title_v4.predict(X_valid_v4_split)
    ),
    1,
)

title_v4_mape = mape_percent(
    y_valid_split,
    y_pred_valid_v4,
)

comparison_v4 = pd.DataFrame(
    {
        "model": [
            "CatBoost base",
            "CatBoost missingness + EV v2",
            "CatBoost age + mileage v3",
            "CatBoost title features v4",
        ],
        "validation_mape_pct": [
            13.433,
            13.410,
            13.407,
            round(title_v4_mape, 3),
        ],
    }
).sort_values("validation_mape_pct").reset_index(drop=True)

display(comparison_v4)
print("Best iteration:", catboost_title_v4.get_best_iteration())

,model,validation_mape_pct
0,CatBoost title features v4,13.272
1,CatBoost age + mileage v3,13.407
2,CatBoost missingness + EV v2,13.410
3,CatBoost base,13.433


Best iteration: 2999


Это не случайная «косметика» уровня 0.02 п.п. как v2/v3. Гипотеза про Полное название сработала: после удаления года модель лучше различает комплектации и похожие автомобили разных лет.

Но один holdout — ещё не финальное доказательство. Теперь нужно сделать 5-fold OOF-проверку v4. Она даст:

честную среднюю CV-метрику;
разброс результатов по фолдам;
OOF-предсказания для будущего нового ансамбля с Ridge.

In [34]:
from sklearn.model_selection import StratifiedKFold

Создаём стратифицированные фолды

In [35]:
cv_target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
).to_numpy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

Запускаем OOF-CV для v4

In [36]:
oof_title_v4_predictions = np.zeros(len(y))
fold_results_v4 = []

for fold, (train_fold_idx, valid_fold_idx) in enumerate(
    cv.split(X_model_v4, cv_target_bins),
    start=1,
):
    print(f"\n{'=' * 60}")
    print(f"Fold {fold}/5")
    print(f"{'=' * 60}")

    X_train_fold = X_model_v4.iloc[train_fold_idx].copy()
    X_valid_fold = X_model_v4.iloc[valid_fold_idx].copy()

    y_train_fold = y.iloc[train_fold_idx].copy()
    y_valid_fold = y.iloc[valid_fold_idx].copy()

    model_fold = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=500,
        allow_writing_files=False,
    )

    model_fold.fit(
        X_train_fold,
        np.log1p(y_train_fold),
        cat_features=categorical_columns_v4,
        eval_set=(
            X_valid_fold,
            np.log1p(y_valid_fold),
        ),
        use_best_model=True,
        early_stopping_rounds=200,
    )

    fold_predictions = np.maximum(
        np.expm1(
            model_fold.predict(X_valid_fold)
        ),
        1,
    )

    oof_title_v4_predictions[valid_fold_idx] = fold_predictions

    fold_mape = mape_percent(
        y_valid_fold,
        fold_predictions,
    )

    fold_results_v4.append(
        {
            "fold": fold,
            "best_iteration": model_fold.get_best_iteration(),
            "validation_mape_pct": fold_mape,
        }
    )

    print(
        f"Fold {fold} MAPE: {fold_mape:.3f}% | "
        f"best iteration: {model_fold.get_best_iteration()}"
    )


Fold 1/5
0:	learn: 0.6539333	test: 0.6461968	best: 0.6461968 (0)	total: 105ms	remaining: 5m 16s
500:	learn: 0.1540691	test: 0.2161252	best: 0.2161252 (500)	total: 49.7s	remaining: 4m 7s
1000:	learn: 0.1137653	test: 0.2081078	best: 0.2080387 (969)	total: 1m 43s	remaining: 3m 26s
1500:	learn: 0.0906985	test: 0.2062337	best: 0.2062171 (1496)	total: 2m 36s	remaining: 2m 36s
2000:	learn: 0.0735111	test: 0.2050202	best: 0.2050104 (1995)	total: 3m 39s	remaining: 1m 49s
2500:	learn: 0.0615190	test: 0.2045990	best: 0.2045652 (2449)	total: 4m 33s	remaining: 54.5s
2999:	learn: 0.0519813	test: 0.2041855	best: 0.2041766 (2989)	total: 5m 48s	remaining: 0us

bestTest = 0.2041766346
bestIteration = 2989

Shrink model to first 2990 iterations.
Fold 1 MAPE: 13.910% | best iteration: 2989

Fold 2/5
0:	learn: 0.6508638	test: 0.6566310	best: 0.6566310 (0)	total: 162ms	remaining: 8m 6s
500:	learn: 0.1585471	test: 0.1953795	best: 0.1953795 (500)	total: 1m 30s	remaining: 7m 30s
1000:	learn: 0.1201144	test: 0

Итоговая CV-оценка

In [37]:
fold_results_v4_df = pd.DataFrame(fold_results_v4)

oof_title_v4_mape = mape_percent(
    y,
    oof_title_v4_predictions,
)

display(fold_results_v4_df)

print(f"OOF MAPE v4: {oof_title_v4_mape:.3f}%")
print(
    "Средний fold MAPE:",
    f"{fold_results_v4_df['validation_mape_pct'].mean():.3f}%"
)
print(
    "Std fold MAPE:",
    f"{fold_results_v4_df['validation_mape_pct'].std():.3f}"
)

,fold,best_iteration,validation_mape_pct
0,1,2989,13.909571
1,2,2980,12.399850
2,3,2959,13.119255
3,4,2999,13.220394
4,5,2996,12.601640


OOF MAPE v4: 13.050%
Средний fold MAPE: 13.050%
Std fold MAPE: 0.591


Сохрани OOF-предсказания

Они понадобятся для нового blend с Ridge.

In [38]:
title_v4_oof_report = pd.DataFrame(
    {
        "car_id": train["car_id"].to_numpy(),
        "y_true": y.to_numpy(),
        "title_v4_pred": oof_title_v4_predictions,
        "ape_pct": (
            np.abs(y.to_numpy() - oof_title_v4_predictions)
            / y.to_numpy()
            * 100
        ),
    }
)

title_v4_oof_report.to_parquet(
    REPORTS_DIR / "catboost_title_v4_oof_predictions.parquet",
    index=False,
)

display(title_v4_oof_report.head())

,car_id,y_true,title_v4_pred,ape_pct
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,47964.148615,23.580719
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,13448.527652,15.683212
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40504.712805,3.537240
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,78713.988302,12.609425
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,30937.398285,10.688366


Теперь главный тест: новый ансамбль Ridge + CatBoost v4.

# ============================================================
# OOF ensemble: Ridge alpha=0.1 + CatBoost title v4
# ============================================================

In [39]:
RIDGE_OOF_PATH = REPORTS_DIR / "ridge_oof_predictions_alpha_0_1.parquet"
TITLE_V4_OOF_PATH = (
    REPORTS_DIR / "catboost_title_v4_oof_predictions.parquet"
)

ridge_oof = pd.read_parquet(RIDGE_OOF_PATH)
title_v4_oof = pd.read_parquet(TITLE_V4_OOF_PATH)

print("Ridge columns:", ridge_oof.columns.tolist())
print("Title v4 columns:", title_v4_oof.columns.tolist())

display(ridge_oof.head())
display(title_v4_oof.head())

Ridge columns: ['car_id', 'y_true', 'ridge_pred']
Title v4 columns: ['car_id', 'y_true', 'title_v4_pred', 'ape_pct']


,car_id,y_true,ridge_pred
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,40566.385555
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,16612.439448
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40958.198793
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,59670.249223
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,27712.258776


,car_id,y_true,title_v4_pred,ape_pct
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,47964.148615,23.580719
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,13448.527652,15.683212
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40504.712805,3.537240
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,78713.988302,12.609425
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,30937.398285,10.688366


Теперь объединяем:

In [40]:
blend_oof = (
    ridge_oof[
        [
            "car_id",
            "y_true",
            "ridge_pred",
        ]
    ]
    .merge(
        title_v4_oof[
            [
                "car_id",
                "y_true",
                "title_v4_pred",
            ]
        ],
        on="car_id",
        how="inner",
        suffixes=("_ridge", "_title"),
        validate="one_to_one",
    )
)

assert len(blend_oof) == len(y), (
    f"Ожидали {len(y)} строк, получили {len(blend_oof)}"
)

assert np.allclose(
    blend_oof["y_true_ridge"],
    blend_oof["y_true_title"],
)

blend_oof["y_true"] = blend_oof["y_true_ridge"]

blend_oof = blend_oof[
    [
        "car_id",
        "y_true",
        "ridge_pred",
        "title_v4_pred",
    ]
].copy()

display(blend_oof.head())

,car_id,y_true,ridge_pred,title_v4_pred
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,40566.385555,47964.148615
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,16612.439448,13448.527652
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40958.198793,40504.712805
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,59670.249223,78713.988302
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,27712.258776,30937.398285


Сначала посмотри метрики отдельных моделей на одной таблице:

In [41]:
individual_oof_results = pd.DataFrame(
    {
        "model": [
            "Ridge log1p alpha=0.1",
            "CatBoost title v4",
        ],
        "oof_mape_pct": [
            mape_percent(
                blend_oof["y_true"],
                blend_oof["ridge_pred"],
            ),
            mape_percent(
                blend_oof["y_true"],
                blend_oof["title_v4_pred"],
            ),
        ],
    }
)

display(individual_oof_results)

,model,oof_mape_pct
0,Ridge log1p alpha=0.1,14.468951
1,CatBoost title v4,13.050142


Теперь ищем оптимальный вес.

In [42]:
# ridge_weight = 0 означает только CatBoost.
# ridge_weight = 1 означает только Ridge.

weights = np.linspace(0, 1, 1001)

blend_search_results = []

for ridge_weight in weights:
    catboost_weight = 1 - ridge_weight

    blend_prediction = (
        ridge_weight * blend_oof["ridge_pred"]
        + catboost_weight * blend_oof["title_v4_pred"]
    )

    blend_mape = mape_percent(
        blend_oof["y_true"],
        blend_prediction,
    )

    blend_search_results.append(
        {
            "ridge_weight": ridge_weight,
            "catboost_weight": catboost_weight,
            "oof_mape_pct": blend_mape,
        }
    )

blend_search_results = (
    pd.DataFrame(blend_search_results)
    .sort_values("oof_mape_pct")
    .reset_index(drop=True)
)

display(blend_search_results.head(15))

,ridge_weight,catboost_weight,oof_mape_pct
0,0.357,0.643,12.407929
1,0.358,0.642,12.407933
2,0.356,0.644,12.407933
3,0.359,0.641,12.407939
4,0.355,0.645,12.407945
5,0.360,0.640,12.407952
6,0.354,0.646,12.407964
7,0.361,0.639,12.407973
8,0.353,0.647,12.407990
9,0.362,0.638,12.407997


Зафиксируй лучший вариант:

In [43]:
best_blend = blend_search_results.iloc[0]

BEST_RIDGE_WEIGHT = float(best_blend["ridge_weight"])
BEST_CATBOOST_WEIGHT = float(best_blend["catboost_weight"])

print(f"Лучший вес Ridge: {BEST_RIDGE_WEIGHT:.3f}")
print(f"Лучший вес CatBoost v4: {BEST_CATBOOST_WEIGHT:.3f}")
print(f"Лучший OOF MAPE: {best_blend['oof_mape_pct']:.3f}%")

Лучший вес Ridge: 0.357
Лучший вес CatBoost v4: 0.643
Лучший OOF MAPE: 12.408%


сохрани OOF-предсказания нового ансамбля:

In [44]:
blend_oof["ensemble_pred"] = (
    BEST_RIDGE_WEIGHT * blend_oof["ridge_pred"]
    + BEST_CATBOOST_WEIGHT * blend_oof["title_v4_pred"]
)

blend_oof["ensemble_ape_pct"] = (
    np.abs(
        blend_oof["y_true"]
        - blend_oof["ensemble_pred"]
    )
    / blend_oof["y_true"]
    * 100
)

blend_oof.to_parquet(
    REPORTS_DIR / "ridge_catboost_title_v4_oof_ensemble.parquet",
    index=False,
)

display(
    blend_oof[
        [
            "car_id",
            "y_true",
            "ridge_pred",
            "title_v4_pred",
            "ensemble_pred",
            "ensemble_ape_pct",
        ]
    ].head()
)

,car_id,y_true,ridge_pred,title_v4_pred,ensemble_pred,ensemble_ape_pct
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,40566.385555,47964.148615,45323.147203,16.776119
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,16612.439448,13448.527652,14578.044163,8.601604
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40958.198793,40504.712805,40666.607303,3.151685
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,59670.249223,78713.988302,71915.373451,2.883224
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,27712.258776,30937.398285,29786.023480,6.568957


In [45]:
RIDGE_WEIGHT = 0.36
CATBOOST_WEIGHT = 0.64

Теперь нужно сделать один обязательный шаг: проверить этот фиксированный ансамбль на том же holdout, который мы не использовали для подбора веса.

Сохрани holdout-предсказания CatBoost v4

In [46]:
title_v4_holdout_report = pd.DataFrame(
    {
        "car_id": train.loc[
            X_valid_v4_split.index,
            "car_id",
        ].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "title_v4_pred": y_pred_valid_v4,
    }
)

title_v4_holdout_report["ape_pct"] = (
    np.abs(
        title_v4_holdout_report["y_true"]
        - title_v4_holdout_report["title_v4_pred"]
    )
    / title_v4_holdout_report["y_true"]
    * 100
)

title_v4_holdout_report.to_parquet(
    REPORTS_DIR / "catboost_title_v4_holdout_predictions.parquet",
    index=False,
)

display(title_v4_holdout_report.head())

,car_id,y_true,title_v4_pred,ape_pct
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,27671.959973,38.429014
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,74437.324197,2.043263
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,32001.381683,7.202025
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,17745.177188,183.922835
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,24858.983493,14.032034


Загрузи Ridge holdout predictions

In [47]:
RIDGE_HOLDOUT_PATH = (
    REPORTS_DIR / "ridge_holdout_predictions_alpha_0_1.parquet"
)

ridge_holdout = pd.read_parquet(RIDGE_HOLDOUT_PATH)

print(ridge_holdout.columns.tolist())
display(ridge_holdout.head())

['car_id', 'y_true', 'ridge_pred']


,car_id,y_true,ridge_pred
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,29376.245745
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,71057.755352
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,33790.401383
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,5180.029616
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,23475.926417


Собери holdout-ансамбль

In [48]:
blend_holdout = (
    ridge_holdout[
        [
            "car_id",
            "y_true",
            "ridge_pred",
        ]
    ]
    .merge(
        title_v4_holdout_report[
            [
                "car_id",
                "y_true",
                "title_v4_pred",
            ]
        ],
        on="car_id",
        how="inner",
        suffixes=("_ridge", "_title"),
        validate="one_to_one",
    )
)

assert len(blend_holdout) == len(y_valid_split)

assert np.allclose(
    blend_holdout["y_true_ridge"],
    blend_holdout["y_true_title"],
)

blend_holdout["y_true"] = blend_holdout["y_true_ridge"]

blend_holdout = blend_holdout[
    [
        "car_id",
        "y_true",
        "ridge_pred",
        "title_v4_pred",
    ]
].copy()

display(blend_holdout.head())

,car_id,y_true,ridge_pred,title_v4_pred
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,29376.245745,27671.959973
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,71057.755352,74437.324197
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,33790.401383,32001.381683
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,5180.029616,17745.177188
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,23475.926417,24858.983493


Оцениваем фиксированный вес 0.36 / 0.64

In [49]:
RIDGE_WEIGHT = 0.36
CATBOOST_WEIGHT = 0.64

blend_holdout["ensemble_pred"] = (
    RIDGE_WEIGHT * blend_holdout["ridge_pred"]
    + CATBOOST_WEIGHT * blend_holdout["title_v4_pred"]
)

holdout_results = pd.DataFrame(
    {
        "model": [
            "Ridge alpha=0.1",
            "CatBoost title v4",
            "Ridge + CatBoost title v4",
        ],
        "holdout_mape_pct": [
            mape_percent(
                blend_holdout["y_true"],
                blend_holdout["ridge_pred"],
            ),
            mape_percent(
                blend_holdout["y_true"],
                blend_holdout["title_v4_pred"],
            ),
            mape_percent(
                blend_holdout["y_true"],
                blend_holdout["ensemble_pred"],
            ),
        ],
    }
).sort_values(
    "holdout_mape_pct"
).reset_index(drop=True)

display(holdout_results)

,model,holdout_mape_pct
0,Ridge + CatBoost title v4,12.340041
1,CatBoost title v4,13.271692
2,Ridge alpha=0.1,14.194180


Финальное обучение CatBoost v4 на всех данных

In [50]:
PROJECT_ROOT = Path.cwd().resolve().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
SUBMISSION_DIR = PROJECT_ROOT / "submission"

MODELS_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

y = train[TARGET_COLUMN].copy()

In [51]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

title_numeric_columns = [
    "Название_число_слов",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
]

In [52]:
X_train_title = add_title_features(
    prepare_features(
        train.drop(columns=[TARGET_COLUMN])
    )
)

X_test_title = add_title_features(
    prepare_features(X_test)
)

feature_columns_v4 = [
    column
    for column in X_train_title.columns
    if column not in EXCLUDED_COLUMNS
]

numeric_columns_v4 = (
    base_numeric_columns
    + title_numeric_columns
)

categorical_columns_v4 = [
    column
    for column in feature_columns_v4
    if column not in numeric_columns_v4
]

X_train_catboost = X_train_title[
    feature_columns_v4
].copy()

X_test_catboost = X_test_title[
    feature_columns_v4
].copy()

for column in categorical_columns_v4:
    X_train_catboost[column] = (
        X_train_catboost[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_catboost[column] = (
        X_test_catboost[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert list(X_train_catboost.columns) == list(X_test_catboost.columns)
assert "car_id" not in X_train_catboost.columns
assert "Предложение" not in X_train_catboost.columns

print(X_train_catboost.shape)
print(X_test_catboost.shape)

(8340, 32)
(8341, 32)


In [53]:
catboost_title_v4_final = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_title_v4_final.fit(
    X_train_catboost,
    np.log1p(y),
    cat_features=categorical_columns_v4,
)

joblib.dump(
    catboost_title_v4_final,
    MODELS_DIR / "catboost_title_v4_3000.joblib",
)

0:	learn: 0.6514888	total: 63.9ms	remaining: 3m 11s
300:	learn: 0.1872452	total: 20.7s	remaining: 3m 5s
600:	learn: 0.1512195	total: 41.7s	remaining: 2m 46s
900:	learn: 0.1285337	total: 1m 2s	remaining: 2m 24s
1200:	learn: 0.1134676	total: 1m 22s	remaining: 2m 3s
1500:	learn: 0.1002732	total: 1m 43s	remaining: 1m 43s
1800:	learn: 0.0889765	total: 2m 5s	remaining: 1m 23s
2100:	learn: 0.0806080	total: 2m 26s	remaining: 1m 2s
2400:	learn: 0.0729806	total: 2m 59s	remaining: 44.7s
2700:	learn: 0.0666305	total: 3m 34s	remaining: 23.8s
2999:	learn: 0.0612307	total: 4m 3s	remaining: 0us


NameError: name 'joblib' is not defined

In [54]:
import joblib

joblib.dump(
    catboost_title_v4_final,
    MODELS_DIR / "catboost_title_v4_3000.joblib",
)

print("Модель сохранена:")
print(MODELS_DIR / "catboost_title_v4_3000.joblib")

Модель сохранена:
C:\temp\shift_ml\models\catboost_title_v4_3000.joblib


Загружаем финальный Ridge и делаем blend

In [57]:
# ------------------------------------------------------------
# Ridge: готовим test ровно в том формате,
# который ожидал сохранённый Pipeline
# ------------------------------------------------------------

X_test_ridge = prepare_features(X_test)

ridge_feature_columns = [
    column
    for column in X_test_ridge.columns
    if column not in EXCLUDED_COLUMNS
]

ridge_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

ridge_categorical_columns = [
    column
    for column in ridge_feature_columns
    if column not in ridge_numeric_columns
]

X_test_ridge_model = X_test_ridge[
    ridge_feature_columns
].copy()

# Числовые признаки: обычный float + np.nan.
for column in ridge_numeric_columns:
    X_test_ridge_model[column] = pd.to_numeric(
        X_test_ridge_model[column],
        errors="coerce",
    ).astype(float)

# Категории: object + np.nan вместо pandas pd.NA.
for column in ridge_categorical_columns:
    X_test_ridge_model[column] = (
        X_test_ridge_model[column]
        .astype(object)
        .where(
            X_test_ridge_model[column].notna(),
            np.nan,
        )
    )

print(X_test_ridge_model.dtypes.value_counts())
print("Есть ли pd.NA:", X_test_ridge_model.isna().sum().sum())

object     13
float64     9
Name: count, dtype: int64
Есть ли pd.NA: 6547


Теперь прогноз:

In [58]:
ridge_predictions = np.maximum(
    ridge_final.predict(X_test_ridge_model),
    1,
)

Для контроля сравни число и порядок признаков, которые ждёт обученный Ridge:

In [59]:
preprocessor = ridge_final.regressor_.named_steps["preprocessor"]

print("Числовые в сохранённой модели:")
print(preprocessor.transformers_[0][2])

print("\nКатегориальные в сохранённой модели:")
print(preprocessor.transformers_[1][2])

print("\nЧисло столбцов на входе:", X_test_ridge_model.shape[1])

Числовые в сохранённой модели:
['Год выпуска', 'Оценка эксперта', 'Количество владельцев', 'Пробег_число', 'Расход_л_на_100км', 'Двигатель_цилиндры', 'Двигатель_объём_л', 'Двери_число', 'Кресла_число']

Категориальные в сохранённой модели:
['Бренд', 'Модель', 'Тип машины', 'Полное название', 'Исползование', 'КПП', 'Двигатель', 'Привод', 'Топливо', 'Цвет', 'Локация', 'Тип кузова', 'Штат']

Число столбцов на входе: 22


Да. Ниже один цельный блок: он не переобучает модели, а загружает обе готовые модели, корректно готовит test для Ridge и CatBoost, делает blend и создаёт submission.csv + .zip.

Замени всё, начиная с ridge_final = ..., на этот код и выполняй ячейки по порядку.

1. Импорты, пути, данные и модели

In [60]:
from pathlib import Path
import zipfile

import joblib
import numpy as np
import pandas as pd

from src.feature_engineering import (
    prepare_features,
    add_title_features,
)

PROJECT_ROOT = Path.cwd().resolve().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
SUBMISSION_DIR = PROJECT_ROOT / "submission"

SUBMISSION_DIR.mkdir(exist_ok=True)

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

ridge_final = joblib.load(
    MODELS_DIR / "ridge_log_target_alpha_0_1.joblib"
)

catboost_title_v4_final = joblib.load(
    MODELS_DIR / "catboost_title_v4_3000.joblib"
)

print("train:", train.shape)
print("X_test:", X_test.shape)

train: (8340, 23)
X_test: (8341, 22)


2. Общие списки исключений и числовых признаков

In [61]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

title_numeric_columns = [
    "Название_число_слов",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
]

3. Test-матрица для CatBoost v4

In [62]:
X_test_title = add_title_features(
    prepare_features(X_test)
)

feature_columns_v4 = [
    column
    for column in X_test_title.columns
    if column not in EXCLUDED_COLUMNS
]

numeric_columns_v4 = (
    base_numeric_columns
    + title_numeric_columns
)

categorical_columns_v4 = [
    column
    for column in feature_columns_v4
    if column not in numeric_columns_v4
]

X_test_catboost = X_test_title[
    feature_columns_v4
].copy()

for column in categorical_columns_v4:
    X_test_catboost[column] = (
        X_test_catboost[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert list(X_test_catboost.columns) == list(
    catboost_title_v4_final.feature_names_
)

print("CatBoost test matrix:", X_test_catboost.shape)

CatBoost test matrix: (8341, 32)


4. Test-матрица для Ridge

Этот блок специально исправляет ошибку с pd.NA.

In [63]:
X_test_ridge = prepare_features(X_test)

# Ridge хранит исходный порядок признаков внутри Pipeline.
ridge_pipeline = ridge_final.regressor_

ridge_feature_columns = list(
    ridge_pipeline.feature_names_in_
)

ridge_numeric_columns = base_numeric_columns

ridge_categorical_columns = [
    column
    for column in ridge_feature_columns
    if column not in ridge_numeric_columns
]

missing_columns = [
    column
    for column in ridge_feature_columns
    if column not in X_test_ridge.columns
]

assert not missing_columns, (
    f"В test отсутствуют колонки Ridge: {missing_columns}"
)

X_test_ridge_model = X_test_ridge[
    ridge_feature_columns
].copy()

# Числа приводим к float, пропуски становятся np.nan.
for column in ridge_numeric_columns:
    X_test_ridge_model[column] = pd.to_numeric(
        X_test_ridge_model[column],
        errors="coerce",
    ).astype(float)

# Категории приводим к object, pd.NA заменяем на обычный np.nan.
for column in ridge_categorical_columns:
    X_test_ridge_model[column] = pd.Series(
        X_test_ridge_model[column]
        .astype("string")
        .to_numpy(dtype=object, na_value=np.nan),
        index=X_test_ridge_model.index,
    )

assert list(X_test_ridge_model.columns) == ridge_feature_columns

print("Ridge test matrix:", X_test_ridge_model.shape)

Ridge test matrix: (8341, 22)


5. Прогнозы и ансамбль

In [64]:
ridge_predictions = np.maximum(
    ridge_final.predict(X_test_ridge_model),
    1,
)

catboost_predictions = np.maximum(
    np.expm1(
        catboost_title_v4_final.predict(X_test_catboost)
    ),
    1,
)

RIDGE_WEIGHT = 0.36
CATBOOST_WEIGHT = 0.64

final_predictions = (
    RIDGE_WEIGHT * ridge_predictions
    + CATBOOST_WEIGHT * catboost_predictions
)

print("Ridge predictions:")
print(pd.Series(ridge_predictions).describe())

print("\nCatBoost predictions:")
print(pd.Series(catboost_predictions).describe())

print("\nFinal blend predictions:")
print(pd.Series(final_predictions).describe())

Ridge predictions:
count    8.341000e+03
mean     3.636137e+04
std      3.677116e+04
min      4.707120e+02
25%      1.918621e+04
50%      2.928936e+04
75%      4.374142e+04
max      1.561766e+06
dtype: float64

CatBoost predictions:
count      8341.000000
mean      36269.909358
std       29907.796745
min        4038.965916
25%       19107.236704
50%       29448.406607
75%       44271.963996
max      577843.099464
dtype: float64

Final blend predictions:
count      8341.000000
mean      36302.836178
std       31504.929514
min        4014.884109
25%       19199.950677
50%       29438.242933
75%       44107.858029
max      932055.360322
dtype: float64


6. Создание submission и архива

In [65]:
submission = pd.DataFrame(
    {
        "Цена": final_predictions,
    }
)

assert submission.shape == (len(X_test), 1)
assert submission.columns.tolist() == ["Цена"]
assert submission["Цена"].notna().all()
assert np.isfinite(submission["Цена"]).all()
assert (submission["Цена"] > 0).all()

csv_path = (
    SUBMISSION_DIR
    / "submission_ridge_catboost_title_v4.csv"
)

submission.to_csv(
    csv_path,
    index=False,
)

zip_path = (
    SUBMISSION_DIR
    / "submission_ridge_catboost_title_v4.zip"
)

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        csv_path,
        arcname="submission.csv",
    )

display(submission.head())

print("CSV:")
print(csv_path)

print("\nZIP:")
print(zip_path)

,Цена
0,29368.211288
1,24658.572695
2,40519.391557
3,54491.414345
4,25918.345953


CSV:
C:\temp\shift_ml\submission\submission_ridge_catboost_title_v4.csv

ZIP:
C:\temp\shift_ml\submission\submission_ridge_catboost_title_v4.zip


Приоритет №1 — сделать title-features более обобщающими

Это мой главный кандидат на реальный прирост.

Сейчас CatBoost получает:

Полное название
Название без года
отдельные флаги SPORT / GT / 4X4 / AWD и т.д.

Но ему всё ещё приходится работать со слишком уникальными строками. Вместо этого нужно дать ему иерархию названия.

Пример:

2019 TOYOTA HILUX SR (4X4)

Префикс 2: TOYOTA HILUX
Префикс 3: TOYOTA HILUX SR
Префикс 4: TOYOTA HILUX SR 4X4

Это очень сильная идея, потому что:

TOYOTA HILUX              → семейство модели
TOYOTA HILUX SR           → комплектация
TOYOTA HILUX SR 4X4       → комплектация + привод

Такие признаки менее разрежены, чем полное название, и лучше переносятся на test.

Перезагрузи модуль

In [72]:
import importlib
import src.feature_engineering as fe

importlib.reload(fe)

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)

Собери X_v5 и проверь, что новые признаки появились

In [73]:
TARGET_COLUMN = "Цена"

X_base = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_v5 = add_title_hierarchy_features(X_base)

y = train[TARGET_COLUMN].copy()

print("X_base:", X_base.shape)
print("X_v5:", X_v5.shape)

X_base: (8340, 29)
X_v5: (8340, 66)


Быстрый аудит новых title-features

Эта ячейка нужна до обучения: убедимся, что флаги не пустые и не сломались.

In [74]:
title_hierarchy_numeric_columns = [
    "Название_мощность_kw",
    "Название_есть_мощность_kw",
    "Название_есть_AMG",
    "Название_есть_M_SPORT",
    "Название_есть_RS",
    "Название_есть_S_LINE",
    "Название_есть_GTI",
    "Название_есть_HSE",
    "Название_есть_SR5",
    "Название_есть_GXL",
    "Название_есть_LIMITED",
    "Название_есть_PREMIUM",
    "Название_есть_COMFORTLINE",
    "Название_есть_ASCENT",
    "Название_есть_ACTIVE",
    "Название_есть_ELITE",
    "Название_есть_TDI",
    "Название_есть_TSI",
    "Название_есть_TFSI",
    "Название_есть_CDI",
    "Название_есть_V6",
    "Название_есть_V8",
]

title_hierarchy_categorical_columns = [
    "Название_нормализованное_без_года",
    "Название_префикс_2",
    "Название_префикс_3",
    "Название_префикс_4",
    "Название_моторный_маркер",
]

feature_counts_v5 = (
    X_v5[title_hierarchy_numeric_columns]
    .sum()
    .sort_values(ascending=False)
    .rename("count")
    .to_frame()
)

display(feature_counts_v5)

for column in title_hierarchy_categorical_columns:
    print(
        f"{column}: "
        f"unique={X_v5[column].nunique(dropna=False)}, "
        f"missing={X_v5[column].isna().sum()}"
    )

display(
    X_v5[
        [
            "Полное название",
            "Название_нормализованное_без_года",
            "Название_префикс_2",
            "Название_префикс_3",
            "Название_префикс_4",
            "Название_мощность_kw",
            "Название_моторный_маркер",
        ]
    ].sample(20, random_state=42)
)

,count
Название_мощность_kw,9594.0
Название_есть_ACTIVE,292.0
Название_есть_ASCENT,205.0
Название_есть_GXL,175.0
Название_есть_TFSI,141.0
Название_есть_TSI,138.0
Название_есть_PREMIUM,127.0
Название_есть_TDI,125.0
Название_есть_ELITE,67.0
Название_есть_RS,63.0


Название_нормализованное_без_года: unique=3032, missing=0
Название_префикс_2: unique=540, missing=0
Название_префикс_3: unique=1596, missing=0
Название_префикс_4: unique=2405, missing=0
Название_моторный_маркер: unique=11, missing=7829


,Полное название,Название_нормализованное_без_года,Название_префикс_2,Название_префикс_3,Название_префикс_4,Название_мощность_kw,Название_моторный_маркер
7470,2015 PORSCHE PANAMERA DIESEL,PORSCHE PANAMERA DIESEL,PORSCHE PANAMERA,PORSCHE PANAMERA DIESEL,PORSCHE PANAMERA DIESEL,NaN,<NA>
586,2023 LEXUS UX200 LUXURY,LEXUS UX200 LUXURY,LEXUS UX200,LEXUS UX200 LUXURY,LEXUS UX200 LUXURY,NaN,<NA>
7961,2018 TOYOTA COROLLA SX (HYBRID),TOYOTA COROLLA SX HYBRID,TOYOTA COROLLA,TOYOTA COROLLA SX,TOYOTA COROLLA SX HYBRID,NaN,<NA>
4405,2020 BMW 220I M SPORT GRAN COUPE,BMW 220I M SPORT GRAN COUPE,BMW 220I,BMW 220I M,BMW 220I M SPORT,NaN,<NA>
5363,2018 VOLKSWAGEN TIGUAN 110 TDI COMFORTLINE,VOLKSWAGEN TIGUAN 110 TDI COMFORTLINE,VOLKSWAGEN TIGUAN,VOLKSWAGEN TIGUAN 110,VOLKSWAGEN TIGUAN 110 TDI,NaN,TDI
1201,2014 AUDI Q3 2.0 TDI QUATTRO (130KW),AUDI Q3 2 0 TDI QUATTRO 130KW,AUDI Q3,AUDI Q3 2,AUDI Q3 2 0,130.0,TDI
7649,2019 TOYOTA HILUX SR (4X4),TOYOTA HILUX SR 4X4,TOYOTA HILUX,TOYOTA HILUX SR,TOYOTA HILUX SR 4X4,NaN,<NA>
879,2013 HOLDEN COMMODORE SS-V REDLINE,HOLDEN COMMODORE SS V REDLINE,HOLDEN COMMODORE,HOLDEN COMMODORE SS,HOLDEN COMMODORE SS V,NaN,<NA>
5885,2019 LAND ROVER DISCOVERY SPORT TD4 (132KW) HS...,LAND ROVER DISCOVERY SPORT TD4 132KW HSE AWD,LAND ROVER,LAND ROVER DISCOVERY,LAND ROVER DISCOVERY SPORT,132.0,TD4
6495,2016 HYUNDAI TUCSON ACTIVE (FWD),HYUNDAI TUCSON ACTIVE FWD,HYUNDAI TUCSON,HYUNDAI TUCSON ACTIVE,HYUNDAI TUCSON ACTIVE FWD,NaN,<NA>


5. Полная подготовка v5 для CatBoost

In [75]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

title_numeric_columns_v4 = [
    "Название_число_слов",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
]

numeric_columns_v5 = (
    base_numeric_columns
    + title_numeric_columns_v4
    + title_hierarchy_numeric_columns
)

feature_columns_v5 = [
    column
    for column in X_v5.columns
    if column not in EXCLUDED_COLUMNS
]

categorical_columns_v5 = [
    column
    for column in feature_columns_v5
    if column not in numeric_columns_v5
]

print("Всего признаков:", len(feature_columns_v5))
print("Числовых:", len(numeric_columns_v5))
print("Категориальных:", len(categorical_columns_v5))

assert set(numeric_columns_v5).issubset(feature_columns_v5)
assert "Название_префикс_2" in categorical_columns_v5
assert "Название_префикс_3" in categorical_columns_v5
assert "Название_префикс_4" in categorical_columns_v5
assert "Предложение" not in feature_columns_v5
assert "car_id" not in feature_columns_v5

Всего признаков: 59
Числовых: 40
Категориальных: 19


Подготовь матрицу и восстанови тот же holdout

In [76]:
from sklearn.model_selection import train_test_split

X_model_v5 = X_v5[
    feature_columns_v5
].copy()

for column in categorical_columns_v5:
    X_model_v5[column] = (
        X_model_v5[column]
        .fillna("__MISSING__")
        .astype(str)
    )

target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
)

train_idx, valid_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    random_state=42,
    stratify=target_bins,
)

X_train_v5_split = X_model_v5.iloc[train_idx].copy()
X_valid_v5_split = X_model_v5.iloc[valid_idx].copy()

y_train_split = y.iloc[train_idx].copy()
y_valid_split = y.iloc[valid_idx].copy()

print("Train:", X_train_v5_split.shape)
print("Validation:", X_valid_v5_split.shape)

Train: (6672, 59)
Validation: (1668, 59)


Обучи CatBoost v5

In [77]:
from catboost import CatBoostRegressor

catboost_title_hierarchy_v5 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_title_hierarchy_v5.fit(
    X_train_v5_split,
    np.log1p(y_train_split),
    cat_features=categorical_columns_v5,
    eval_set=(
        X_valid_v5_split,
        np.log1p(y_valid_split),
    ),
    use_best_model=True,
    early_stopping_rounds=200,
)

0:	learn: 0.6537164	test: 0.6509433	best: 0.6509433 (0)	total: 132ms	remaining: 6m 37s
300:	learn: 0.1805152	test: 0.2131696	best: 0.2131696 (300)	total: 27.6s	remaining: 4m 7s
600:	learn: 0.1448457	test: 0.1982628	best: 0.1982505 (599)	total: 55.6s	remaining: 3m 41s
900:	learn: 0.1233822	test: 0.1930260	best: 0.1930260 (900)	total: 1m 24s	remaining: 3m 16s
1200:	learn: 0.1084648	test: 0.1904151	best: 0.1903900 (1193)	total: 1m 53s	remaining: 2m 50s
1500:	learn: 0.0960088	test: 0.1888664	best: 0.1888664 (1500)	total: 2m 23s	remaining: 2m 23s
1800:	learn: 0.0860286	test: 0.1881375	best: 0.1881039 (1793)	total: 2m 54s	remaining: 1m 55s
2100:	learn: 0.0770348	test: 0.1877267	best: 0.1877239 (2099)	total: 3m 24s	remaining: 1m 27s
2400:	learn: 0.0693669	test: 0.1871492	best: 0.1871453 (2399)	total: 3m 55s	remaining: 58.6s
2700:	learn: 0.0629042	test: 0.1870186	best: 0.1869618 (2681)	total: 4m 25s	remaining: 29.4s
2999:	learn: 0.0570113	test: 0.1866776	best: 0.1866776 (2999)	total: 4m 56s	re

CatBoostRegressor(allow_writing_files=False, depth=8, iterations=3000, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=300)

Сравнение с v4

In [78]:
from sklearn.metrics import mean_absolute_percentage_error

def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


y_pred_valid_v5 = np.maximum(
    np.expm1(
        catboost_title_hierarchy_v5.predict(
            X_valid_v5_split
        )
    ),
    1,
)

title_v5_mape = mape_percent(
    y_valid_split,
    y_pred_valid_v5,
)

comparison_v5 = pd.DataFrame(
    {
        "model": [
            "CatBoost base",
            "CatBoost title features v4",
            "CatBoost title hierarchy v5",
        ],
        "validation_mape_pct": [
            13.433,
            13.272,
            round(title_v5_mape, 3),
        ],
    }
).sort_values(
    "validation_mape_pct"
).reset_index(drop=True)

display(comparison_v5)

print(
    "Best iteration:",
    catboost_title_hierarchy_v5.get_best_iteration(),
)

,model,validation_mape_pct
0,CatBoost title hierarchy v5,13.045
1,CatBoost title features v4,13.272
2,CatBoost base,13.433


Best iteration: 2999


In [79]:
RIDGE_WEIGHT = 0.36
CATBOOST_WEIGHT = 0.64

В этот раз сделаем fast-track: не ждём 5-fold CV, но перед финальным обучением быстро проверяем ансамбль на уже готовом holdout.

Вес не подбираем заново по holdout, чтобы не переобучиться ещё сильнее. Берём устойчивый прежний вес:

Быстрая проверка holdout-ансамбля v5

In [80]:
# ------------------------------------------------------------
# Holdout check: Ridge + CatBoost v5
# ------------------------------------------------------------

RIDGE_WEIGHT = 0.36
CATBOOST_WEIGHT = 0.64

title_v5_holdout = pd.DataFrame(
    {
        "car_id": train.loc[
            X_valid_v5_split.index,
            "car_id",
        ].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "title_v5_pred": y_pred_valid_v5,
    }
)

ridge_holdout = pd.read_parquet(
    REPORTS_DIR / "ridge_holdout_predictions_alpha_0_1.parquet"
)

blend_holdout_v5 = (
    ridge_holdout[
        ["car_id", "y_true", "ridge_pred"]
    ]
    .merge(
        title_v5_holdout[
            ["car_id", "y_true", "title_v5_pred"]
        ],
        on="car_id",
        how="inner",
        suffixes=("_ridge", "_v5"),
        validate="one_to_one",
    )
)

assert len(blend_holdout_v5) == len(y_valid_split)

assert np.allclose(
    blend_holdout_v5["y_true_ridge"],
    blend_holdout_v5["y_true_v5"],
)

blend_holdout_v5["y_true"] = blend_holdout_v5["y_true_ridge"]

blend_holdout_v5["ensemble_pred"] = (
    RIDGE_WEIGHT * blend_holdout_v5["ridge_pred"]
    + CATBOOST_WEIGHT * blend_holdout_v5["title_v5_pred"]
)

holdout_v5_results = pd.DataFrame(
    {
        "model": [
            "Ridge alpha=0.1",
            "CatBoost title hierarchy v5",
            "Ridge + CatBoost title hierarchy v5",
        ],
        "holdout_mape_pct": [
            mape_percent(
                blend_holdout_v5["y_true"],
                blend_holdout_v5["ridge_pred"],
            ),
            mape_percent(
                blend_holdout_v5["y_true"],
                blend_holdout_v5["title_v5_pred"],
            ),
            mape_percent(
                blend_holdout_v5["y_true"],
                blend_holdout_v5["ensemble_pred"],
            ),
        ],
    }
).sort_values("holdout_mape_pct").reset_index(drop=True)

display(holdout_v5_results)

,model,holdout_mape_pct
0,Ridge + CatBoost title hierarchy v5,12.125276
1,CatBoost title hierarchy v5,13.044887
2,Ridge alpha=0.1,14.194180


2. Финальный CatBoost v5 на всех данных

In [81]:
# ------------------------------------------------------------
# Final CatBoost v5: train on all available training objects
# ------------------------------------------------------------

import joblib

X_train_v5_final = add_title_hierarchy_features(
    prepare_features(
        train.drop(columns=[TARGET_COLUMN])
    )
)

X_test_v5_final = add_title_hierarchy_features(
    prepare_features(X_test)
)

feature_columns_v5 = [
    column
    for column in X_train_v5_final.columns
    if column not in EXCLUDED_COLUMNS
]

X_train_catboost_v5 = X_train_v5_final[
    feature_columns_v5
].copy()

X_test_catboost_v5 = X_test_v5_final[
    feature_columns_v5
].copy()

for column in categorical_columns_v5:
    X_train_catboost_v5[column] = (
        X_train_catboost_v5[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_catboost_v5[column] = (
        X_test_catboost_v5[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert list(X_train_catboost_v5.columns) == list(
    X_test_catboost_v5.columns
)

print("Train matrix:", X_train_catboost_v5.shape)
print("Test matrix:", X_test_catboost_v5.shape)

Train matrix: (8340, 59)
Test matrix: (8341, 59)


In [82]:
catboost_title_hierarchy_v5_final = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_title_hierarchy_v5_final.fit(
    X_train_catboost_v5,
    np.log1p(y),
    cat_features=categorical_columns_v5,
)

joblib.dump(
    catboost_title_hierarchy_v5_final,
    MODELS_DIR / "catboost_title_hierarchy_v5_3000.joblib",
)

0:	learn: 0.6522414	total: 83.5ms	remaining: 4m 10s
300:	learn: 0.1835397	total: 39.6s	remaining: 5m 54s
600:	learn: 0.1481288	total: 1m 29s	remaining: 5m 58s
900:	learn: 0.1272140	total: 1m 57s	remaining: 4m 33s
1200:	learn: 0.1120938	total: 2m 27s	remaining: 3m 40s
1500:	learn: 0.0999856	total: 3m 1s	remaining: 3m 1s
1800:	learn: 0.0900040	total: 3m 30s	remaining: 2m 20s
2100:	learn: 0.0815609	total: 3m 59s	remaining: 1m 42s
2400:	learn: 0.0748068	total: 4m 27s	remaining: 1m 6s
2700:	learn: 0.0683762	total: 4m 56s	remaining: 32.9s
2999:	learn: 0.0630450	total: 5m 33s	remaining: 0us


['C:\\temp\\shift_ml\\models\\catboost_title_hierarchy_v5_3000.joblib']

Прогноз Ridge

Используй рабочий блок, который мы уже сделали для Ridge с заменой pd.NA → np.nan:

In [83]:
# ------------------------------------------------------------
# Ridge test matrix
# ------------------------------------------------------------

ridge_final = joblib.load(
    MODELS_DIR / "ridge_log_target_alpha_0_1.joblib"
)

X_test_ridge = prepare_features(X_test)

ridge_pipeline = ridge_final.regressor_

ridge_feature_columns = list(
    ridge_pipeline.feature_names_in_
)

ridge_numeric_columns = base_numeric_columns

ridge_categorical_columns = [
    column
    for column in ridge_feature_columns
    if column not in ridge_numeric_columns
]

X_test_ridge_model = X_test_ridge[
    ridge_feature_columns
].copy()

for column in ridge_numeric_columns:
    X_test_ridge_model[column] = pd.to_numeric(
        X_test_ridge_model[column],
        errors="coerce",
    ).astype(float)

for column in ridge_categorical_columns:
    X_test_ridge_model[column] = pd.Series(
        X_test_ridge_model[column]
        .astype("string")
        .to_numpy(dtype=object, na_value=np.nan),
        index=X_test_ridge_model.index,
    )

ridge_predictions = np.maximum(
    ridge_final.predict(X_test_ridge_model),
    1,
)

4. Blend и submission

In [84]:
# ------------------------------------------------------------
# Final v5 blend and submission
# ------------------------------------------------------------

catboost_v5_predictions = np.maximum(
    np.expm1(
        catboost_title_hierarchy_v5_final.predict(
            X_test_catboost_v5
        )
    ),
    1,
)

RIDGE_WEIGHT = 0.36
CATBOOST_WEIGHT = 0.64

final_predictions_v5 = (
    RIDGE_WEIGHT * ridge_predictions
    + CATBOOST_WEIGHT * catboost_v5_predictions
)

submission_v5 = pd.DataFrame(
    {
        "Цена": final_predictions_v5,
    }
)

assert submission_v5.shape == (len(X_test), 1)
assert submission_v5.columns.tolist() == ["Цена"]
assert submission_v5["Цена"].notna().all()
assert np.isfinite(submission_v5["Цена"]).all()
assert (submission_v5["Цена"] > 0).all()

csv_path_v5 = (
    SUBMISSION_DIR
    / "submission_ridge_catboost_title_hierarchy_v5.csv"
)

zip_path_v5 = (
    SUBMISSION_DIR
    / "submission_ridge_catboost_title_hierarchy_v5.zip"
)

submission_v5.to_csv(
    csv_path_v5,
    index=False,
)

with zipfile.ZipFile(
    zip_path_v5,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        csv_path_v5,
        arcname="submission.csv",
    )

display(submission_v5.head())

print("CSV:", csv_path_v5)
print("ZIP:", zip_path_v5)

,Цена
0,31016.571698
1,24884.978782
2,40541.751936
3,56543.925450
4,24905.307599


CSV: C:\temp\shift_ml\submission\submission_ridge_catboost_title_hierarchy_v5.csv
ZIP: C:\temp\shift_ml\submission\submission_ridge_catboost_title_hierarchy_v5.zip


Делаем v6: убираем только сырой Полное название, но сохраняем все признаки, которые мы из него извлекли:

Название_без_года
Название_нормализованное_без_года
Название_префикс_2 / 3 / 4
мощность KW
моторные маркеры
флаги комплектаций

Так мы проверим гипотезу: не переобучается ли CatBoost на почти уникальных полных строках.

Ничего в feature_engineering.py менять не нужно.

1. Собираем v6

In [85]:
# ============================================================
# Experiment v6:
# title hierarchy without raw "Полное название"
# ============================================================

TARGET_COLUMN = "Цена"

X_base = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_v6 = add_title_hierarchy_features(X_base)

y = train[TARGET_COLUMN].copy()

print("X_v6:", X_v6.shape)

X_v6: (8340, 66)


In [86]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

# Единственное отличие v6 от v5:
# исключаем сырой почти уникальный title.
EXCLUDED_COLUMNS_V6 = EXCLUDED_COLUMNS + [
    "Полное название",
]

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

title_numeric_columns_v4 = [
    "Название_число_слов",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
]

title_hierarchy_numeric_columns = [
    "Название_мощность_kw",
    "Название_есть_мощность_kw",
    "Название_есть_AMG",
    "Название_есть_M_SPORT",
    "Название_есть_RS",
    "Название_есть_S_LINE",
    "Название_есть_GTI",
    "Название_есть_HSE",
    "Название_есть_SR5",
    "Название_есть_GXL",
    "Название_есть_LIMITED",
    "Название_есть_PREMIUM",
    "Название_есть_COMFORTLINE",
    "Название_есть_ASCENT",
    "Название_есть_ACTIVE",
    "Название_есть_ELITE",
    "Название_есть_TDI",
    "Название_есть_TSI",
    "Название_есть_TFSI",
    "Название_есть_CDI",
    "Название_есть_V6",
    "Название_есть_V8",
]

numeric_columns_v6 = (
    base_numeric_columns
    + title_numeric_columns_v4
    + title_hierarchy_numeric_columns
)

feature_columns_v6 = [
    column
    for column in X_v6.columns
    if column not in EXCLUDED_COLUMNS_V6
]

categorical_columns_v6 = [
    column
    for column in feature_columns_v6
    if column not in numeric_columns_v6
]

print("Всего признаков:", len(feature_columns_v6))
print("Числовых:", len(numeric_columns_v6))
print("Категориальных:", len(categorical_columns_v6))

assert "Полное название" not in feature_columns_v6
assert "Название_без_года" in feature_columns_v6
assert "Название_префикс_2" in feature_columns_v6
assert "Название_префикс_3" in feature_columns_v6
assert "Название_префикс_4" in feature_columns_v6

Всего признаков: 58
Числовых: 40
Категориальных: 18


Матрица и тот же holdout

In [87]:
X_model_v6 = X_v6[
    feature_columns_v6
].copy()

for column in categorical_columns_v6:
    X_model_v6[column] = (
        X_model_v6[column]
        .fillna("__MISSING__")
        .astype(str)
    )

# Используем уже существующие train_idx и valid_idx.
X_train_v6_split = X_model_v6.iloc[train_idx].copy()
X_valid_v6_split = X_model_v6.iloc[valid_idx].copy()

y_train_split = y.iloc[train_idx].copy()
y_valid_split = y.iloc[valid_idx].copy()

print("Train:", X_train_v6_split.shape)
print("Validation:", X_valid_v6_split.shape)

Train: (6672, 58)
Validation: (1668, 58)


Обучение

In [88]:
catboost_title_hierarchy_v6 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_title_hierarchy_v6.fit(
    X_train_v6_split,
    np.log1p(y_train_split),
    cat_features=categorical_columns_v6,
    eval_set=(
        X_valid_v6_split,
        np.log1p(y_valid_split),
    ),
    use_best_model=True,
    early_stopping_rounds=200,
)

0:	learn: 0.6538509	test: 0.6509994	best: 0.6509994 (0)	total: 69.7ms	remaining: 3m 29s
300:	learn: 0.1842528	test: 0.2163073	best: 0.2163073 (300)	total: 26.6s	remaining: 3m 58s
600:	learn: 0.1467579	test: 0.2005478	best: 0.2005478 (600)	total: 52.4s	remaining: 3m 29s
900:	learn: 0.1251166	test: 0.1945632	best: 0.1945626 (891)	total: 1m 18s	remaining: 3m 3s
1200:	learn: 0.1104496	test: 0.1916116	best: 0.1916061 (1199)	total: 1m 44s	remaining: 2m 37s
1500:	learn: 0.0982768	test: 0.1898236	best: 0.1897616 (1489)	total: 2m 11s	remaining: 2m 11s
1800:	learn: 0.0884546	test: 0.1885928	best: 0.1885648 (1797)	total: 2m 44s	remaining: 1m 49s
2100:	learn: 0.0792673	test: 0.1878812	best: 0.1878767 (2098)	total: 3m 33s	remaining: 1m 31s
2400:	learn: 0.0717217	test: 0.1874077	best: 0.1874072 (2398)	total: 4m 22s	remaining: 1m 5s
2700:	learn: 0.0648738	test: 0.1868865	best: 0.1868711 (2680)	total: 5m 9s	remaining: 34.3s
2999:	learn: 0.0585626	test: 0.1864078	best: 0.1864078 (2999)	total: 5m 48s	re

CatBoostRegressor(allow_writing_files=False, depth=8, iterations=3000, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=300)

Сравнение v5 и v6

In [89]:
y_pred_valid_v6 = np.maximum(
    np.expm1(
        catboost_title_hierarchy_v6.predict(
            X_valid_v6_split
        )
    ),
    1,
)

title_v6_mape = mape_percent(
    y_valid_split,
    y_pred_valid_v6,
)

comparison_v6 = pd.DataFrame(
    {
        "model": [
            "CatBoost title features v4",
            "CatBoost title hierarchy v5",
            "CatBoost hierarchy without raw title v6",
        ],
        "validation_mape_pct": [
            13.272,
            13.045,
            round(title_v6_mape, 3),
        ],
    }
).sort_values(
    "validation_mape_pct"
).reset_index(drop=True)

display(comparison_v6)

print(
    "Best iteration:",
    catboost_title_hierarchy_v6.get_best_iteration(),
)

,model,validation_mape_pct
0,CatBoost hierarchy without raw title v6,12.909
1,CatBoost title hierarchy v5,13.045
2,CatBoost title features v4,13.272


Best iteration: 2999


Полное название действительно, похоже, переобучало CatBoost: оно слишком уникальное, а иерархические признаки сохраняют полезный смысл без привязки к конкретной строке.

Теперь не подбираем новый вес на этом же holdout. Берём уже подтверждённый вес:

RIDGE_WEIGHT = 0.36
CATBOOST_WEIGHT = 0.64

И быстро проверяем ансамбль v6 с Ridge.

In [90]:
# ============================================================
# Fast holdout blend: Ridge + CatBoost v6
# ============================================================

RIDGE_WEIGHT = 0.36
CATBOOST_WEIGHT = 0.64

title_v6_holdout = pd.DataFrame(
    {
        "car_id": train.loc[
            X_valid_v6_split.index,
            "car_id",
        ].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "title_v6_pred": y_pred_valid_v6,
    }
)

ridge_holdout = pd.read_parquet(
    REPORTS_DIR / "ridge_holdout_predictions_alpha_0_1.parquet"
)

blend_holdout_v6 = (
    ridge_holdout[
        ["car_id", "y_true", "ridge_pred"]
    ]
    .merge(
        title_v6_holdout[
            ["car_id", "y_true", "title_v6_pred"]
        ],
        on="car_id",
        how="inner",
        suffixes=("_ridge", "_v6"),
        validate="one_to_one",
    )
)

assert len(blend_holdout_v6) == len(y_valid_split)

assert np.allclose(
    blend_holdout_v6["y_true_ridge"],
    blend_holdout_v6["y_true_v6"],
)

blend_holdout_v6["y_true"] = blend_holdout_v6["y_true_ridge"]

blend_holdout_v6["ensemble_pred"] = (
    RIDGE_WEIGHT * blend_holdout_v6["ridge_pred"]
    + CATBOOST_WEIGHT * blend_holdout_v6["title_v6_pred"]
)

holdout_v6_results = pd.DataFrame(
    {
        "model": [
            "Ridge alpha=0.1",
            "CatBoost hierarchy without raw title v6",
            "Ridge + CatBoost hierarchy v6",
        ],
        "holdout_mape_pct": [
            mape_percent(
                blend_holdout_v6["y_true"],
                blend_holdout_v6["ridge_pred"],
            ),
            mape_percent(
                blend_holdout_v6["y_true"],
                blend_holdout_v6["title_v6_pred"],
            ),
            mape_percent(
                blend_holdout_v6["y_true"],
                blend_holdout_v6["ensemble_pred"],
            ),
        ],
    }
).sort_values(
    "holdout_mape_pct"
).reset_index(drop=True)

display(holdout_v6_results)

,model,holdout_mape_pct
0,Ridge + CatBoost hierarchy v6,12.146338
1,CatBoost hierarchy without raw title v6,12.908705
2,Ridge alpha=0.1,14.194180


У v6 CatBoost сам по себе лучше:

v5 CatBoost: 13.045%
v6 CatBoost: 12.909%

Но в ансамбле с фиксированным весом 0.36 / 0.64 v6 чуть хуже:

v5 blend: 12.125%
v6 blend: 12.146%
разница: 0.021 п.п.

0.021 — очень маленькая разница. Она не доказывает, что v5 лучше. Скорее всего, у v6 ошибки немного изменились: он стал сильнее сам, но может сильнее пересекаться по ошибкам с Ridge. Тогда оптимальный вес Ridge уже не обязан быть 0.36.

Но подбирать новый вес прямо по этому holdout опасно: мы начнём переобучаться на валидации. Поэтому предлагаю сделать быстрый диагностический поиск, не принимать его как финальную истину.

In [91]:
weights = np.arange(0.10, 0.61, 0.01)

weight_diagnostic_v6 = []

for ridge_weight in weights:
    catboost_weight = 1 - ridge_weight

    pred = (
        ridge_weight * blend_holdout_v6["ridge_pred"]
        + catboost_weight * blend_holdout_v6["title_v6_pred"]
    )

    weight_diagnostic_v6.append(
        {
            "ridge_weight": round(ridge_weight, 2),
            "catboost_weight": round(catboost_weight, 2),
            "holdout_mape_pct": mape_percent(
                blend_holdout_v6["y_true"],
                pred,
            ),
        }
    )

weight_diagnostic_v6 = (
    pd.DataFrame(weight_diagnostic_v6)
    .sort_values("holdout_mape_pct")
    .reset_index(drop=True)
)

display(weight_diagnostic_v6.head(15))

,ridge_weight,catboost_weight,holdout_mape_pct
0,0.39,0.61,12.144051
1,0.38,0.62,12.144229
2,0.40,0.60,12.144690
3,0.37,0.63,12.145076
4,0.41,0.59,12.146172
5,0.36,0.64,12.146338
6,0.42,0.58,12.148373
7,0.35,0.65,12.148727
8,0.43,0.57,12.151322
9,0.34,0.66,12.151766


Самый дешёвый и логичный следующий шаг — проверить ансамбль из трёх прогнозов на том же holdout:

Ridge + CatBoost v5 + CatBoost v6

v5 и v6 строят похожие, но не идентичные ошибки: v5 использует raw title, v6 — нет. Иногда их усреднение даёт выигрыш даже тогда, когда каждая модель по отдельности неидеальна.

In [92]:
# ============================================================
# Holdout diagnostic:
# Ridge + CatBoost v5 + CatBoost v6
# ============================================================

# Восстанавливаем holdout-предсказания v5 и v6.

title_v5_holdout = pd.DataFrame(
    {
        "car_id": train.loc[
            X_valid_v5_split.index,
            "car_id",
        ].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "title_v5_pred": y_pred_valid_v5,
    }
)

title_v6_holdout = pd.DataFrame(
    {
        "car_id": train.loc[
            X_valid_v6_split.index,
            "car_id",
        ].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "title_v6_pred": y_pred_valid_v6,
    }
)

ridge_holdout = pd.read_parquet(
    REPORTS_DIR / "ridge_holdout_predictions_alpha_0_1.parquet"
)

blend_three_holdout = (
    ridge_holdout[
        ["car_id", "y_true", "ridge_pred"]
    ]
    .merge(
        title_v5_holdout[
            ["car_id", "y_true", "title_v5_pred"]
        ],
        on="car_id",
        how="inner",
        suffixes=("_ridge", "_v5"),
        validate="one_to_one",
    )
    .merge(
        title_v6_holdout[
            ["car_id", "y_true", "title_v6_pred"]
        ],
        on="car_id",
        how="inner",
        validate="one_to_one",
    )
)

assert len(blend_three_holdout) == len(y_valid_split)

assert np.allclose(
    blend_three_holdout["y_true_ridge"],
    blend_three_holdout["y_true_v5"],
)

assert np.allclose(
    blend_three_holdout["y_true_ridge"],
    blend_three_holdout["y_true"],
)

blend_three_holdout["y_true"] = (
    blend_three_holdout["y_true_ridge"]
)

blend_three_holdout = blend_three_holdout[
    [
        "car_id",
        "y_true",
        "ridge_pred",
        "title_v5_pred",
        "title_v6_pred",
    ]
].copy()

display(blend_three_holdout.head())

,car_id,y_true,ridge_pred,title_v5_pred,title_v6_pred
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,29376.245745,29572.410203,27689.867035
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,71057.755352,79194.898436,76247.848754
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,33790.401383,32772.532724,32363.511722
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,5180.029616,18901.033937,17564.474856
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,23475.926417,23949.139002,23929.230906


Теперь ищем веса с шагом 0.01:

In [93]:
weight_search_three_models = []

for ridge_weight in np.arange(0, 1.01, 0.01):
    for v5_weight in np.arange(0, 1.01 - ridge_weight, 0.01):
        v6_weight = 1 - ridge_weight - v5_weight

        prediction = (
            ridge_weight * blend_three_holdout["ridge_pred"]
            + v5_weight * blend_three_holdout["title_v5_pred"]
            + v6_weight * blend_three_holdout["title_v6_pred"]
        )

        weight_search_three_models.append(
            {
                "ridge_weight": round(ridge_weight, 2),
                "v5_weight": round(v5_weight, 2),
                "v6_weight": round(v6_weight, 2),
                "holdout_mape_pct": mape_percent(
                    blend_three_holdout["y_true"],
                    prediction,
                ),
            }
        )

weight_search_three_models = (
    pd.DataFrame(weight_search_three_models)
    .sort_values("holdout_mape_pct")
    .reset_index(drop=True)
)

display(weight_search_three_models.head(20))

,ridge_weight,v5_weight,v6_weight,holdout_mape_pct
0,0.38,0.35,0.27,12.080835
1,0.38,0.36,0.26,12.080913
2,0.37,0.36,0.27,12.081014
3,0.37,0.35,0.28,12.081026
4,0.38,0.34,0.28,12.081029
5,0.37,0.37,0.26,12.081122
6,0.37,0.34,0.29,12.081197
7,0.38,0.37,0.25,12.081201
8,0.38,0.33,0.29,12.081350
9,0.37,0.38,0.25,12.081425


И для сравнения выведи три базовых варианта:

In [94]:
reference_blends = pd.DataFrame(
    {
        "model": [
            "Ridge + v5 (0.36 / 0.64)",
            "Ridge + v6 (0.39 / 0.61)",
            "Ridge + v5 + v6, equal CatBoost split",
        ],
        "holdout_mape_pct": [
            mape_percent(
                blend_three_holdout["y_true"],
                0.36 * blend_three_holdout["ridge_pred"]
                + 0.64 * blend_three_holdout["title_v5_pred"],
            ),
            mape_percent(
                blend_three_holdout["y_true"],
                0.39 * blend_three_holdout["ridge_pred"]
                + 0.61 * blend_three_holdout["title_v6_pred"],
            ),
            mape_percent(
                blend_three_holdout["y_true"],
                0.36 * blend_three_holdout["ridge_pred"]
                + 0.32 * blend_three_holdout["title_v5_pred"]
                + 0.32 * blend_three_holdout["title_v6_pred"],
            ),
        ],
    }
).sort_values("holdout_mape_pct").reset_index(drop=True)

display(reference_blends)

,model,holdout_mape_pct
0,"Ridge + v5 + v6, equal CatBoost split",12.083062
1,Ridge + v5 (0.36 / 0.64),12.125276
2,Ridge + v6 (0.39 / 0.61),12.144051


Это хороший сигнал: v5 и v6 действительно дополняют друг друга.

Ridge + v5:          12.125%
Ridge + v6:          12.144%
Ridge + v5 + v6:     12.081%

Прирост над лучшим двойным ансамблем:

12.125% → 12.081%
−0.044 п.п.

Это небольшой, но реальный локальный выигрыш. И он не держится на одном сверхточном наборе весов: рядом много почти одинаковых комбинаций.

Я бы не брал точные оптимальные 0.38 / 0.35 / 0.27, потому что они подогнаны под один holdout.

Возьмём более устойчивые и понятные веса:

RIDGE_WEIGHT = 0.38
V5_WEIGHT = 0.33
V6_WEIGHT = 0.29

Они близки к оптимуму, но не слишком агрессивно предпочитают v5.

Перед финальным обучением можно быстро проверить именно этот вариант:

In [95]:
stable_three_blend_pred = (
    0.38 * blend_three_holdout["ridge_pred"]
    + 0.33 * blend_three_holdout["title_v5_pred"]
    + 0.29 * blend_three_holdout["title_v6_pred"]
)

print(
    "Stable three-model blend MAPE:",
    f"{mape_percent(blend_three_holdout['y_true'], stable_three_blend_pred):.3f}%"
)

Stable three-model blend MAPE: 12.081%


Важно: текущую v6-модель с holdout использовать нельзя — она обучалась только на 80% train. Ниже код обучит v6 на всех данных, загрузит финальные Ridge и v5, соберёт тройной ансамбль и создаст zip.

In [96]:
from pathlib import Path
import zipfile

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)

PROJECT_ROOT = Path.cwd().resolve().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
SUBMISSION_DIR = PROJECT_ROOT / "submission"

MODELS_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

y = train[TARGET_COLUMN].copy()

print("train:", train.shape)
print("X_test:", X_test.shape)

train: (8340, 23)
X_test: (8341, 22)


Общие списки признаков

In [97]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

EXCLUDED_COLUMNS_V6 = EXCLUDED_COLUMNS + [
    "Полное название",
]

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

title_numeric_columns_v4 = [
    "Название_число_слов",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
]

title_hierarchy_numeric_columns = [
    "Название_мощность_kw",
    "Название_есть_мощность_kw",
    "Название_есть_AMG",
    "Название_есть_M_SPORT",
    "Название_есть_RS",
    "Название_есть_S_LINE",
    "Название_есть_GTI",
    "Название_есть_HSE",
    "Название_есть_SR5",
    "Название_есть_GXL",
    "Название_есть_LIMITED",
    "Название_есть_PREMIUM",
    "Название_есть_COMFORTLINE",
    "Название_есть_ASCENT",
    "Название_есть_ACTIVE",
    "Название_есть_ELITE",
    "Название_есть_TDI",
    "Название_есть_TSI",
    "Название_есть_TFSI",
    "Название_есть_CDI",
    "Название_есть_V6",
    "Название_есть_V8",
]

numeric_columns_v5_v6 = (
    base_numeric_columns
    + title_numeric_columns_v4
    + title_hierarchy_numeric_columns
)

Подготовка v5 и v6

In [98]:
X_train_title = add_title_hierarchy_features(
    prepare_features(
        train.drop(columns=[TARGET_COLUMN])
    )
)

X_test_title = add_title_hierarchy_features(
    prepare_features(X_test)
)

# v5: raw "Полное название" остаётся.
feature_columns_v5 = [
    column
    for column in X_train_title.columns
    if column not in EXCLUDED_COLUMNS
]

categorical_columns_v5 = [
    column
    for column in feature_columns_v5
    if column not in numeric_columns_v5_v6
]

# v6: raw "Полное название" исключаем.
feature_columns_v6 = [
    column
    for column in X_train_title.columns
    if column not in EXCLUDED_COLUMNS_V6
]

categorical_columns_v6 = [
    column
    for column in feature_columns_v6
    if column not in numeric_columns_v5_v6
]

print("v5 features:", len(feature_columns_v5))
print("v6 features:", len(feature_columns_v6))

assert "Полное название" in feature_columns_v5
assert "Полное название" not in feature_columns_v6

v5 features: 59
v6 features: 58


Загружаем Ridge и финальный v5

In [99]:
ridge_model_path = (
    MODELS_DIR / "ridge_log_target_alpha_0_1.joblib"
)

v5_model_path = (
    MODELS_DIR / "catboost_title_hierarchy_v5_3000.joblib"
)

assert ridge_model_path.exists(), (
    f"Не найден Ridge: {ridge_model_path}"
)

assert v5_model_path.exists(), (
    f"Не найдена финальная v5-модель: {v5_model_path}"
)

ridge_final = joblib.load(ridge_model_path)

catboost_v5_final = joblib.load(v5_model_path)

print("Ridge загружен")
print("v5 загружена")

Ridge загружен
v5 загружена


5. Готовим test для v5

In [100]:
# Берём порядок колонок прямо из сохранённой модели.
v5_model_features = list(
    catboost_v5_final.feature_names_
)

missing_v5_columns = [
    column
    for column in v5_model_features
    if column not in X_test_title.columns
]

assert not missing_v5_columns, (
    f"Не хватает колонок для v5: {missing_v5_columns}"
)

X_test_v5 = X_test_title[
    v5_model_features
].copy()

v5_categorical_columns = [
    column
    for column in v5_model_features
    if column not in numeric_columns_v5_v6
]

for column in v5_categorical_columns:
    X_test_v5[column] = (
        X_test_v5[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert list(X_test_v5.columns) == v5_model_features

print("X_test_v5:", X_test_v5.shape)

X_test_v5: (8341, 59)


Обучаем финальную v6 на всех train-данных

In [101]:
X_train_v6 = X_train_title[
    feature_columns_v6
].copy()

X_test_v6 = X_test_title[
    feature_columns_v6
].copy()

for column in categorical_columns_v6:
    X_train_v6[column] = (
        X_train_v6[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_v6[column] = (
        X_test_v6[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert list(X_train_v6.columns) == list(X_test_v6.columns)

catboost_v6_final = CatBoostRegressor(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=300,
    allow_writing_files=False,
)

catboost_v6_final.fit(
    X_train_v6,
    np.log1p(y),
    cat_features=categorical_columns_v6,
)

v6_model_path = (
    MODELS_DIR / "catboost_title_hierarchy_v6_3000.joblib"
)

joblib.dump(
    catboost_v6_final,
    v6_model_path,
)

print("v6 сохранена:")
print(v6_model_path)

0:	learn: 0.6531931	total: 97.1ms	remaining: 4m 51s
300:	learn: 0.1829689	total: 34.9s	remaining: 5m 12s
600:	learn: 0.1465959	total: 1m 16s	remaining: 5m 4s
900:	learn: 0.1261943	total: 1m 59s	remaining: 4m 37s
1200:	learn: 0.1117519	total: 2m 25s	remaining: 3m 38s
1500:	learn: 0.0999679	total: 2m 52s	remaining: 2m 52s
1800:	learn: 0.0895382	total: 3m 19s	remaining: 2m 13s
2100:	learn: 0.0815915	total: 3m 46s	remaining: 1m 37s
2400:	learn: 0.0742428	total: 4m 13s	remaining: 1m 3s
2700:	learn: 0.0680814	total: 4m 47s	remaining: 31.9s
2999:	learn: 0.0624081	total: 5m 15s	remaining: 0us
v6 сохранена:
C:\temp\shift_ml\models\catboost_title_hierarchy_v6_3000.joblib


Готовим test для Ridge

In [102]:
X_test_ridge = prepare_features(X_test)

ridge_pipeline = ridge_final.regressor_

ridge_feature_columns = list(
    ridge_pipeline.feature_names_in_
)

ridge_numeric_columns = base_numeric_columns

ridge_categorical_columns = [
    column
    for column in ridge_feature_columns
    if column not in ridge_numeric_columns
]

missing_ridge_columns = [
    column
    for column in ridge_feature_columns
    if column not in X_test_ridge.columns
]

assert not missing_ridge_columns, (
    f"Не хватает колонок для Ridge: {missing_ridge_columns}"
)

X_test_ridge_model = X_test_ridge[
    ridge_feature_columns
].copy()

for column in ridge_numeric_columns:
    X_test_ridge_model[column] = pd.to_numeric(
        X_test_ridge_model[column],
        errors="coerce",
    ).astype(float)

for column in ridge_categorical_columns:
    X_test_ridge_model[column] = pd.Series(
        X_test_ridge_model[column]
        .astype("string")
        .to_numpy(
            dtype=object,
            na_value=np.nan,
        ),
        index=X_test_ridge_model.index,
    )

assert list(X_test_ridge_model.columns) == ridge_feature_columns

print("X_test_ridge:", X_test_ridge_model.shape)

X_test_ridge: (8341, 22)


Предсказания и тройной ансамбль

In [103]:
ridge_predictions = np.maximum(
    ridge_final.predict(X_test_ridge_model),
    1,
)

v5_predictions = np.maximum(
    np.expm1(
        catboost_v5_final.predict(X_test_v5)
    ),
    1,
)

v6_predictions = np.maximum(
    np.expm1(
        catboost_v6_final.predict(X_test_v6)
    ),
    1,
)

RIDGE_WEIGHT = 0.38
V5_WEIGHT = 0.33
V6_WEIGHT = 0.29

assert np.isclose(
    RIDGE_WEIGHT + V5_WEIGHT + V6_WEIGHT,
    1.0,
)

final_predictions = (
    RIDGE_WEIGHT * ridge_predictions
    + V5_WEIGHT * v5_predictions
    + V6_WEIGHT * v6_predictions
)

print("Ridge:")
print(pd.Series(ridge_predictions).describe())

print("\nv5:")
print(pd.Series(v5_predictions).describe())

print("\nv6:")
print(pd.Series(v6_predictions).describe())

print("\nTriple blend:")
print(pd.Series(final_predictions).describe())

Ridge:
count    8.341000e+03
mean     3.636137e+04
std      3.677116e+04
min      4.707120e+02
25%      1.918621e+04
50%      2.928936e+04
75%      4.374142e+04
max      1.561766e+06
dtype: float64

v5:
count      8341.000000
mean      36344.281899
std       30368.810237
min        4080.738003
25%       19174.407716
50%       29380.714171
75%       44138.673346
max      539563.610243
dtype: float64

v6:
count      8341.000000
mean      36269.669995
std       30263.641215
min        4222.912272
25%       18991.361951
50%       29348.551376
75%       44064.676145
max      572291.851762
dtype: float64

Triple blend:
count      8341.000000
mean      36329.138969
std       31837.227887
min        4078.993162
25%       19190.289294
50%       29400.269030
75%       43977.817252
max      924842.897582
dtype: float64


Создаём CSV и ZIP

In [104]:
submission = pd.DataFrame(
    {
        "Цена": final_predictions,
    }
)

assert submission.shape == (len(X_test), 1)
assert submission.columns.tolist() == ["Цена"]
assert submission["Цена"].notna().all()
assert np.isfinite(submission["Цена"]).all()
assert (submission["Цена"] > 0).all()

csv_path = (
    SUBMISSION_DIR
    / "submission_ridge_v5_v6_triple.csv"
)

zip_path = (
    SUBMISSION_DIR
    / "submission_ridge_v5_v6_triple.zip"
)

submission.to_csv(
    csv_path,
    index=False,
)

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        csv_path,
        arcname="submission.csv",
    )

with zipfile.ZipFile(zip_path, "r") as archive:
    print("Содержимое ZIP:", archive.namelist())

display(submission.head())

print("\nCSV:", csv_path)
print("ZIP:", zip_path)

Содержимое ZIP: ['submission.csv']


,Цена
0,31008.662192
1,25199.359944
2,41447.571698
3,56080.188845
4,25072.312316



CSV: C:\temp\shift_ml\submission\submission_ridge_v5_v6_triple.csv
ZIP: C:\temp\shift_ml\submission\submission_ridge_v5_v6_triple.zip


Идём последовательно: сначала выясним, насколько hidden test похож на train по названиям и моделям. Это определит, какие признаки реально способны переноситься, а не только улучшать знакомый holdout.

Начинаем с train/test audit по названиям. Сейчас ничего не обучаем: сначала проверим, какие уровни названия реально повторяются в test.

In [105]:
# Если REPORTS_DIR ещё не определён:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

In [106]:
# Сохраняем честные holdout-прогнозы v5.

v5_holdout_report = pd.DataFrame(
    {
        "car_id": train.iloc[valid_idx]["car_id"].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "title_v5_pred": y_pred_valid_v5,
    }
)

v5_holdout_report.to_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v5_holdout.parquet",
    index=False,
)

display(v5_holdout_report.head())

,car_id,y_true,title_v5_pred
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,29572.410203
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,79194.898436
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,32772.532724
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,18901.033937
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,23949.139002


In [107]:
# Сохраняем честные holdout-прогнозы v6.

v6_holdout_report = pd.DataFrame(
    {
        "car_id": train.iloc[valid_idx]["car_id"].to_numpy(),
        "y_true": y_valid_split.to_numpy(),
        "title_v6_pred": y_pred_valid_v6,
    }
)

v6_holdout_report.to_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v6_holdout.parquet",
    index=False,
)

display(v6_holdout_report.head())

,car_id,y_true,title_v6_pred
0,c8308aaa-236b-4d69-b6ed-1656a0da72a7,19990,27689.867035
1,0d0cc3b5-b9a2-4417-92ef-5bf6975467e5,75990,76247.848754
2,ad545ff1-51f5-486c-9293-0c5ee10a7bcd,34485,32363.511722
3,37f7453f-ef82-428d-9bc4-f73d9689b194,6250,17564.474856
4,6a9a3d1a-dc59-4498-8f8d-23c7be368461,21800,23929.230906
